# 04. Fluctuation Diagnostics for Plasma Perturbations and Transient Events

Session 03 built the background state: an equilibrium you can interrogate, and the kinetic
profiles mapped onto it. This session reads what lives *on top of* that state,

$$
X(\mathbf{x},t) = X_0(\mathbf{x},t) + \delta X(\mathbf{x},t),
$$

and asks how stationary laboratory diagnostics see a perturbation that rotates, grows and ends.

```text
Session 03   background equilibrium / kinetic state
      ↓
Session 04   measured perturbations and transient dynamics      <- here
      ↓
Session 05   stability / response / perturbed-equilibrium modelling
```

The path through it is

$$
\text{perturbation} \rightarrow \text{diagnostic response} \rightarrow \text{signal representation}
\rightarrow \text{frequency / phase / spatial structure} \rightarrow \text{multi-diagnostic interpretation}.
$$

The design of the session follows issue [#1005](https://github.com/VEST-Tokamak/vaft/issues/1005).

## Session Overview

**The question.** How do stationary laboratory diagnostics observe rotating and evolving plasma
perturbations, and how can measurements from several diagnostics be combined to identify a mode
or a transient event?

By the end of this session you will be able to:

- explain $m$, $n$ and $\omega$ for a rotating helical perturbation, and tell the laboratory frame
  from the plasma frame — an observed frequency is not an intrinsic one;
- say what each VEST fluctuation diagnostic actually measures ($\delta B$, $\delta\epsilon$,
  $\delta\int n_e\,dl$, line emission) and how sampling, Nyquist frequency, sensor response,
  exposure and line integration limit its usable bandwidth;
- read a Mirnov spectrogram, track a coherent frequency $f(t)$, measure band power against a
  measured floor, and fit a toroidal mode number **with its alias step**;
- find the same perturbation in soft X-ray chords and in a 50 kframe/s camera, and decide whether
  two diagnostics saw *the same mode* (coherence and phase) or merely *the same moment*;
- connect a measured $(f, n)$ to plasma rotation and to the equilibrium's rational surfaces, and
  state where that chain stops;
- describe a current quench as a measured sequence of numbers, without naming it.

**The discharges.** No archived VEST shot carries a toroidal Mirnov array, soft X-ray and a fast
camera together, so the session uses each shot for what it has:

| shot | role | what it brings |
| --- | --- | --- |
| 45531 | `SHOT`, the main case | outboard fluctuation Mirnov array at three toroidal positions (2 MHz / 500 kHz), 64 equilibrium probes at 250 kHz, 52 soft X-ray chords at 976.6 kHz, $I_p$, H-alpha, Langmuir probes; **no equilibrium** |
| 40600 | `CAMERA_SHOT` | 602 fast-camera frames at 50 kframe/s beside the outboard probe the camera papers use |
| 39915 | `EQUILIBRIUM_SHOT` | the EFIT equilibrium of sessions 02 and 03, for $q$ and rational surfaces; a probe set that **cannot** fit $n$ |
| 48224 | `ROTATION_SHOT` | charge-exchange toroidal velocity, for an order-of-magnitude Doppler estimate |
| 41524, 41672 | Part II | two more 250 kHz discharges for the multi-shot comparison |

45531 and 40600 are repository-only samples: they load from a source checkout, not from an
installed wheel. Every quantity is always labelled with the shot it came from; a number from one
shot is never silently used on another.

**For method depth.** Welch's resolution-against-variance trade, spectral breaks, FIR against IIR
filtering, group delay and the $dB/dt \rightarrow B$ transfer are the subject of
`notebooks/fluctuation_diagnostics_analysis.ipynb`. This session is physics- and
diagnostic-oriented and links there rather than repeating it.

**The central message.** A plasma perturbation is not identified by a spectral peak alone. It is
inferred by combining frequency, spatial phase, diagnostic response, plasma rotation, equilibrium
structure and consistency across diagnostics.

## Physical Context

### 1. From equilibrium to perturbation

Write a quantity in flux coordinates as the equilibrium part plus a perturbation,

$$
X(\psi,\theta,\phi,t) = X_0(\psi) + \tilde X(\psi,\theta,\phi,t),
\qquad
\tilde X = A(\psi)\cos\!\left(m\theta - n\phi - \omega t + \varphi_0\right).
$$

- $m$ — poloidal structure: how many times the phase turns going once the short way round;
- $n$ — toroidal structure: the same, the long way round;
- $\omega = 2\pi f$ — how fast the pattern moves past a fixed point;
- $A(\psi)$ — how strong it is, and where; $\varphi_0$ — where the pattern sits.

**Frequency and mode number are different projections of the same rotating spatial structure.**
A single fixed probe sees only the pattern passing it — a frequency. Several probes at different
angles sample its *spatial* phase — a mode number.

### 2. Laboratory frame and plasma frame

A pattern carried by a moving plasma is Doppler shifted:

$$
\omega_{\mathrm{lab}} = \omega_{\mathrm{plasma}} + \mathbf{k}\cdot\mathbf{V}_0
\;\sim\; \omega_{\mathrm{plasma}} + n\,\Omega_\phi - m\,\Omega_\theta ,
$$

with the signs set by the chosen convention. When toroidal rotation dominates,

$$
f_{\mathrm{lab}} \approx f_{\mathrm{plasma}} + n\, f_\phi, \qquad f_\phi = \frac{v_\phi}{2\pi R}.
$$

Three contrasting cases:

- **A rotating tearing mode** is a magnetic island that sits on a rational surface and is largely
  carried by the plasma there, so its lab frequency is mostly $n f_\phi$ of that surface. The
  observed frequency is then a *rotation* measurement as much as a mode property.
- **A locked mode** has $f_{\mathrm{lab}} \rightarrow 0$. Zero observed frequency is not zero
  perturbation — it is a perturbation that stopped moving past the probes, which a pickup coil
  (it measures $dB/dt$) then stops seeing.
- **An Alfvén eigenmode** has an intrinsic plasma-frame frequency set by the Alfvén speed, often
  well above the rotation term, so its lab frequency is not a convection frequency at all.

### 3. How diagnostics observe a rotating mode

Two different kinds of instrument are involved:

- **background-rotation diagnostics** (ion Doppler spectroscopy, charge-exchange spectroscopy)
  measure $v_\phi$ or $\Omega_\phi$ of the plasma itself;
- **fluctuation diagnostics** (Mirnov probes, soft X-ray, fast camera) measure a signal $\delta X(t)$
  the perturbation produces as it passes.

For probes separated toroidally, $\Delta\varphi \sim n\,\Delta\phi$; separated poloidally,
$\Delta\varphi \sim m\,\Delta\theta$ (in a straight-field-line angle, not the geometric one). The
endpoint the whole session works toward is

$$
\boxed{\,f_{\mathrm{lab}} + (m, n) + \Omega_{\phi,\theta} \;\rightarrow\; f_{\mathrm{plasma}}\,}
$$

and each of the three inputs has to be *measured*, not assumed.

### 4. What fluctuates in a tokamak

The table is **typical order of magnitude guidance, not a mode-identification rule**. A feature in
the "tearing" band is not thereby a tearing mode: the band depends on the machine, the rotation
and the mode.

| phenomenon | character | typical observed scale |
| --- | --- | --- |
| locked / slowly rotating MHD | coherent, near-stationary | ~0 to a few kHz |
| tearing mode / NTM | coherent rotating island | a few to tens of kHz |
| fishbone / energetic-particle MHD | burst, chirping | tens of kHz |
| Alfvén eigenmodes | coherent, chirping | tens to hundreds of kHz, or higher |
| IRE precursor | rotating low-$n$ MHD and harmonics | kHz to tens of kHz |
| IRE | nonlinear reconnection event | broadband, transient |
| major disruption | precursor plus nonlinear event | not one frequency |

A **mode** can be tracked through $f(t)$, $m$ and $n$. An **event** — an IRE, a disruption — is a
sequence:

```text
coherent precursor -> growth / coupling -> spectral broadening -> nonlinear event
                   -> current / radiation / edge response -> recovery or termination
```

### 5. What a pickup coil measures

A Mirnov coil measures the flux change through its winding:

$$
V(t) \propto \frac{dB}{dt}
\qquad\Longrightarrow\qquad
S_{dB/dt}(f) = (2\pi f)^2\, S_B(f).
$$

A spectral index fitted to raw pickup voltage is therefore the field's index **plus two**. VAFT
does not integrate behind an axis label: a spectrum axis that says `V^2/Hz` means it. It also
means a slowly rotating or locked perturbation all but disappears from the voltage.

### 6. What an array can resolve

Around the torus a coherent perturbation has $\varphi_{\text{phase}} = \varphi_0 - n\,\phi$. Phases
are read modulo a turn, so if the probes sit at angles spaced by $\Delta\phi$, every $n$ differing
by $360^\circ/\Delta\phi$ predicts the same phase at every probe. Three positions 90 degrees apart
give $n$ **modulo 4**; two positions 120 degrees apart give $n$ modulo 3. State the modulus, and
the number of distinct positions, or you have not stated the result.

### 7. Transients, and what VAFT will not call them

VAFT has no IRE, sawtooth, ELM or disruption **classifier**. It has measurements:
`vaft.process.transients.current_quench` (the 80–20 % time and the steepest $dI_p/dt$) and
`current_spike` (a positive excursion of $|I_p|$ above its trend), each returning numbers and,
when nothing qualifies, a `reason` — never a label. So this session says *onset*, *quench*,
*spike* and *termination*, and names nothing it cannot support.

## Load / Prepare Data

### The fluctuation discharge

`SHOT = 45531`, the VEST demonstration discharge for fluctuation work. The sample is
repository-only, so this cell needs a source checkout; the commented line is the lab-mode path for
any shot. Probes and chords are always selected **by name**: an index depends on which archive
fields a shot happened to carry.

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

import vaft
from vaft.omas.plasma_timing import plasma_timing
from vaft.omas.discharge_timing import discharge_timing
from vaft.process.fluctuation import (
    compute_band_power, compute_psd, compute_spectrogram, cross_spectrum,
    fit_power_law_spectrum, track_dominant_frequency,
)
from vaft.process.transients import current_quench, current_spike
from vaft.process import camera_fluctuation, soft_x_rays
from vaft.process.magnetics import toroidal_phase_fit_at_time
from vaft.machine_mapping.magnetics import toroidal_array_for_shot

In [ ]:
SHOT = 45531
ods = vaft.omas.sample_ods(SHOT)
# ods = vaft.database.load(SHOT)  # lab mode: any shot, needs HSDS credentials

probe_names = [str(ods[f"magnetics.b_field_pol_probe.{i}.name"])
               for i in range(len(ods["magnetics.b_field_pol_probe"]))]
has_sxr = "soft_x_rays" in ods
sxr_names = ([str(ods[f"soft_x_rays.channel.{i}.name"]) for i in range(len(ods["soft_x_rays.channel"]))]
             if has_sxr else [])
print(f"shot {SHOT}: {', '.join(sorted(ods.keys()))}")
print(f"{len(probe_names)} magnetic probes, {len(sxr_names)} soft X-ray chords, "
      f"equilibrium stored: {'equilibrium' in ods}")

### The companion discharges

Each loads from the packaged samples. They are named once here and referred to by their constant
everywhere else, so no cell mixes one shot's numbers into another's without saying so.

In [ ]:
CAMERA_SHOT = 40600        # 50 kframe/s camera beside the outboard probe
EQUILIBRIUM_SHOT = 39915   # the EFIT equilibrium of sessions 02 and 03
ROTATION_SHOT = 48224      # charge-exchange toroidal velocity

camera_ods = vaft.omas.sample_ods(CAMERA_SHOT)
reference = vaft.omas.sample_ods(EQUILIBRIUM_SHOT)
rotation = vaft.omas.sample_ods(ROTATION_SHOT)
for shot, case in ((CAMERA_SHOT, camera_ods), (EQUILIBRIUM_SHOT, reference), (ROTATION_SHOT, rotation)):
    print(f"shot {shot}: {', '.join(sorted(case.keys()))}")

### When was there a plasma, and what fired first

Every window in this session comes from the timing helpers, never from a typed time: the plasma
window from `plasma_timing`, the pre-discharge floor from the first coil onset in
`discharge_timing`.

In [ ]:
timing = plasma_timing(ods)
events = discharge_timing(ods)
ip_time = np.asarray(ods["magnetics.ip.0.time"], dtype=float)
ip = np.asarray(ods["magnetics.ip.0.data"], dtype=float)

window = (timing.onset, timing.offset)
floor_end = events.oh_onset if events.oh_onset is not None else timing.onset
floor_window = (float(ip_time[0]), floor_end)
i_peak = int(np.argmax(np.abs(ip)))

print(f"plasma window : {window[0] * 1e3:.2f} - {window[1] * 1e3:.2f} ms  "
      f"(source {timing.source}, agreement {timing.agreement})")
print(f"OH onset      : {floor_end * 1e3:.2f} ms ({events.oh_coil}); "
      f"pre-discharge floor {floor_window[0] * 1e3:.2f} - {floor_window[1] * 1e3:.2f} ms")
print(f"peak |Ip|     : {abs(ip[i_peak]) / 1e3:.1f} kA at {ip_time[i_peak] * 1e3:.2f} ms")

### What each record can resolve

Section 5 of the context, measured rather than recalled. `fluctuation_bandwidths` reads every
channel's own stored time base, so a record decimated when it was written reports the rate it was
written at, not the digitiser's nominal one.

In [ ]:
bandwidths = vaft.omas.fluctuation_bandwidths(ods)
camera_bandwidths = vaft.omas.fluctuation_bandwidths(camera_ods)

print(f"{'shot':>6}  {'diagnostic':38s} {'sample rate':>13} {'Nyquist':>11}")
for shot, table in ((SHOT, bandwidths), (CAMERA_SHOT, camera_bandwidths)):
    for label, (rate, nyquist) in table.items():
        print(f"{shot:>6}  {label:38s} {rate / 1e3:10.1f} kHz {nyquist / 1e3:8.1f} kHz")
for label in ("Interferometer", "FAST camera", "Soft X-ray"):
    if label not in bandwidths:
        print(f"{SHOT}: no {label} record in this input")

In [ ]:
coverage = {f"{label} #{SHOT}": value for label, value in bandwidths.items()}
coverage.update({f"{label} #{CAMERA_SHOT}": value for label, value in camera_bandwidths.items()
                 if label == "FAST camera"})
exposure = float(camera_ods["camera_visible.channel.0.detector.0.exposure_time"])
coverage[f"camera exposure 1/(2 t_exp) #{CAMERA_SHOT}"] = 0.5 / exposure

figure, axes = vaft.plot.plot_fluctuation_frequency_coverage(
    diagnostics=coverage, phenomena=vaft.plot.fluctuation.TYPICAL_PHENOMENON_BANDS,
)
plt.show()

Read the bars as ceilings. A diagnostic's bar ends at its Nyquist frequency, and
$f_{\mathrm{usable}} < f_{\mathrm{Nyquist}}$: the sensor transfer function, analogue and anti-alias
filtering, the digitiser response, the exposure time, the signal-to-noise ratio, spatial averaging
and line-of-sight integration all cut the usable band further. The camera row shows one of those
limits explicitly: a 19 µs exposure averages the light over about a fifth of a 10 kHz period.
Sampling rate alone is never detectability.

The top half is the phenomenon table of the context, drawn on the same axis, and its caption says
what the text said: order of magnitude, not an identification rule.

### The perturbation-diagnostics map

Different diagnostics do not measure the same perturbation variable — $\delta B$, $\delta n_e$,
$\delta T_e$, $\delta\epsilon$ and line emission are different quantities, integrated over
different volumes.

| diagnostic | perturbation quantity | spatial information | typical role |
| --- | --- | --- | --- |
| Mirnov / magnetic probes | $\delta B$, measured as $dB/dt$ | local; poloidal and toroidal arrays | frequency, phase, mode number |
| soft X-ray | emissivity fluctuation $\delta\epsilon$ | line-of-sight chords | core MHD structure |
| fast camera | visible-emission fluctuation | projected 2-D image | spatial mode imaging |
| interferometer | $\delta\int n_e\,dl$ | line-integrated | density fluctuation |
| Langmuir / triple probe | local edge $n_e$, $T_e$ | local | edge fluctuations |
| H-alpha / UV filterscope | line emission | line-of-sight | onset, envelope, edge activity |
| hard X-ray | energetic-electron radiation | detector dependent | runaway / disruption transients |

Which of them this session's inputs actually carry:

In [ ]:
for ids in ("magnetics", "soft_x_rays", "camera_visible", "interferometer", "langmuir_probes",
            "spectrometer_uv", "hard_x_rays", "charge_exchange", "equilibrium"):
    carried = [str(shot) for shot, case in ((SHOT, ods), (CAMERA_SHOT, camera_ods),
                                             (EQUILIBRIUM_SHOT, reference), (ROTATION_SHOT, rotation))
               if ids in case]
    print(f"{ids:16s} {', '.join(carried) if carried else 'none of these inputs'}")

## Guided Analysis

### Step 1 — Mirnov probes: magnetic fluctuations

#### 1.1 Where the probes are

The 64 equilibrium probes all sit at one toroidal angle; the outboard fluctuation array adds
probes at three. The poloidal cross-section shows *where around the minor radius* each probe is;
the top view shows *where around the torus*, which is what a toroidal mode number needs.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 6))
vaft.omas.plot_magnetics_geometry_poloidal(ods, ax=axes[0])
vaft.omas.plot_machine_geometry_topview(ods, ax=axes[1])
figure.tight_layout()
plt.show()

array = toroidal_array_for_shot(SHOT)
print(f"toroidal array on shot {SHOT}: {array['name']}, IMAS angles {array['angles_deg']} deg, "
      f"alias step {array['alias_step']}")
silent = [name for i, name in enumerate(probe_names)
          if "voltage" in ods[f"magnetics.b_field_pol_probe.{i}"]
          and np.ptp(np.asarray(ods[f"magnetics.b_field_pol_probe.{i}.voltage.data"])) == 0.0]
print(f"probes whose stored voltage is identically flat: {silent or 'none'}")

#### 1.2 Raw traces

An inboard probe, the outboard equilibrium probe `C2-05` (250 kHz), a probe low on the vessel,
and the outboard fluctuation probe at the midplane (2 MHz). `layout="subplots"` gives each its
own scale; the axis says `V` because it is a voltage, $\propto dB/dt$.

In [ ]:
eq_probe = "MagneticFieldProbe_C2-05_Bz"                 # outboard equilibrium probe, 250 kHz
fluct_probe = "OutMirnov_45_L1-03"                       # outboard fluctuation array, midplane
mode_probe = fluct_probe if fluct_probe in probe_names else eq_probe
selected = [probe_names.index(name) for name in
            ("MagneticFieldProbe_H2-05_Bz", eq_probe, "MagneticFieldProbe_L-04_Bz", mode_probe)]
vaft.omas.plot_mirnov_time_voltage(ods, channels=selected, x_limits=window, layout="subplots")
plt.show()

#### 1.3 A spectrogram, and a tracked frequency

`track=` draws the dominant ridge the spectrogram itself contains, found by
`track_dominant_frequency`: a search band, a floor below which a window reports nothing, and a
continuity limit `max_jump` between windows. The ridge is a measurement of the drawn map, not a
model of the mode.

In [ ]:
mode_index = probe_names.index(mode_probe)
figure, axes = vaft.omas.plot_mirnov_spectrogram(
    ods, channel=mode_index, method="stft", time_range=(floor_window[0], window[1]),
    nperseg=2048, max_frequency=6.0e4, track=(2.0e3, 4.0e4), max_jump=3.0e3,
)
for when in window:
    axes.axvline(when, color="k", ls="--", lw=0.8)
plt.show()

mode_time = np.asarray(ods[f"magnetics.b_field_pol_probe.{mode_index}.voltage.time"], dtype=float)
mode_voltage = np.asarray(ods[f"magnetics.b_field_pol_probe.{mode_index}.voltage.data"], dtype=float)
mode_evolution = compute_spectrogram(mode_time, mode_voltage, window_duration=1.0e-3, overlap=0.5)
mode_track = track_dominant_frequency(mode_evolution, search_range=(2.0e3, 4.0e4), max_jump=3.0e3)
tracked = np.isfinite(mode_track.frequency)
if tracked.any():
    t_mode = float(mode_track.time[np.nanargmax(np.where(tracked, mode_track.power, np.nan))])
    f_mode = float(mode_track.frequency[np.argmin(np.abs(mode_track.time - t_mode))])
    print(f"tracked windows: {tracked.sum()} of {tracked.size}, "
          f"{mode_track.time[tracked][0] * 1e3:.1f} - {mode_track.time[tracked][-1] * 1e3:.1f} ms")
    print(f"tracked frequency range: {np.nanmin(mode_track.frequency) / 1e3:.1f} - "
          f"{np.nanmax(mode_track.frequency) / 1e3:.1f} kHz")
    print(f"strongest tracked window: {t_mode * 1e3:.2f} ms at {f_mode / 1e3:.1f} kHz")
else:
    # Nothing cleared the tracker's floor: no line to fit phases at.  The phase
    # steps below are drawn at the window's middle and say they have no frequency.
    t_mode, f_mode = 0.5 * (window[0] + window[1]), float("nan")
    print(f"shot {SHOT}: no window of {mode_probe} cleared the tracker's floor in 2-40 kHz -- "
          f"no tracked line, so no tracked frequency for the phase fits below")

Read it left to right. Nothing structured before the coils fire; a coherent line appearing
after the onset, near 7 kHz with a harmonic near 14 kHz; then a drop to a lower branch around
3–4 kHz while the current peaks and begins to fall; broadband activity near the offset. That is an
**onset**, a **frequency change**, and a **termination** — three measured features, none of them
named. Whether the drop is the mode slowing down, the plasma slowing down, or a different mode
taking over cannot be decided from one probe; the rest of the session collects the evidence
that could.

In [ ]:
eq_index = probe_names.index(eq_probe)
figure, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
vaft.omas.plot_mirnov_spectrogram(ods, channel=eq_index, method="stft", time_range=window,
                                  nperseg=512, max_frequency=4.0e4, ax=axes[0])
vaft.omas.plot_mirnov_spectrogram(ods, channel=eq_index, method="hann_fft", time_range=window,
                                  window_size=500, max_frequency=4.0e4, ax=axes[1])
axes[0].set_title(f'{eq_probe}: method="stft"')
axes[1].set_title(f'{eq_probe}: method="hann_fft"')
axes[0].set_xlabel("")
plt.show()

#### 1.4 One signal, two transforms

`method=` selects scipy's short-time Fourier transform or VAFT's Hann-window FFT, kept because the
VEST Mirnov analyses were written against it. Both get a window of about 2 ms here, so switching
method changes the algorithm and not, silently, the resolution. A window of duration $T$ resolves
$\Delta f \approx 1/T$ and blurs anything faster than $T$: a feature that survives both transforms
is in the signal; a feature that appears in only one is in the transform.

In [ ]:
vaft.omas.plot_mirnov_spectrum(
    ods, channel=mode_index, time_range=window, nperseg=8192,
    fit_ranges=[(2.0e3, 2.0e4), (3.0e4, 3.0e5)], series_label=f"{mode_probe} voltage",
)
plt.show()

inside = (mode_time >= window[0]) & (mode_time <= window[1])
spectrum = compute_psd(mode_time[inside], mode_voltage[inside], nperseg=8192)
for low, high in ((2.0e3, 2.0e4), (3.0e4, 3.0e5)):
    fit = fit_power_law_spectrum(spectrum.frequency, spectrum.psd, f_range=(low, high))
    print(f"{low / 1e3:6.0f}-{high / 1e3:4.0f} kHz : alpha(dB/dt) = {fit.alpha:+.2f}  "
          f"R^2 = {fit.r_squared:.3f}  ->  alpha(B) = {fit.alpha - 2.0:+.2f}")

#### 1.5 A spectral index, and what it is an index of

The y axis says `V^2/Hz`, so both indices are $dB/dt$'s; the `alpha(B)` column is the subtraction
from the context and nothing more. $R^2$ is part of the result: the low band contains the
coherent lines of the spectrogram, a power law is the wrong model there, and the fit says so with
a near-zero $R^2$. Quote the high-band index, as a $dB/dt$ index, or move the band — never quote a
slope whose fit rejected it. VAFT draws no reference slopes of its own.

In [ ]:
band_power = np.array([compute_band_power(mode_evolution.frequency, column, {"mhd": (2.0e3, 4.0e4)})["mhd"]
                       for column in mode_evolution.magnitude.T])
quiet = np.median(band_power[(mode_evolution.time >= floor_window[0]) & (mode_evolution.time < floor_window[1])])
active = np.median(band_power[(mode_evolution.time >= window[0]) & (mode_evolution.time <= window[1])])

figure, axes = plt.subplots(figsize=(8, 4))
axes.semilogy(mode_evolution.time * 1e3, band_power)
axes.axvspan(floor_window[0] * 1e3, floor_window[1] * 1e3, color="0.8", label="pre-discharge floor")
axes.axvspan(window[0] * 1e3, window[1] * 1e3, alpha=0.15, label="plasma window")
axes.axhline(quiet, color="k", ls=":", lw=0.8)
axes.set_xlabel("Time [ms]")
axes.set_ylabel(f"2-40 kHz band power [V$^2$]\n{mode_probe}")
axes.legend()
axes.grid(alpha=0.5)
plt.show()

print(f"band power before the coils fired  : {quiet:.2e}")
print(f"band power inside the plasma window: {active:.2e}  ({active / quiet:.0f}x)")
print("fluctuation power rises above the pre-discharge floor:", bool(active > 10.0 * quiet))

#### 1.6 A transient is a change measured against a floor

The floor is not zero: before the coils fire the band already carries pickup. Measuring the rise
against *that* is the content of the claim, which is why the printed number is a ratio.
`compute_band_power` integrates the band a reader named and attaches no meaning to its name.

#### 1.7 The toroidal mode number

Draw the measured phases first, with no line through them: nothing appears on a VAFT plot that the
caller did not ask for. The time is the strongest tracked window of 1.3, found from the data.

In [ ]:
try:
    vaft.omas.plot_mirnov_spatial_phase(ods, time=t_mode, num_modes=1, candidate_n=range(0, 7), show_fit=False)
    plt.show()
except ValueError as error:   # a shot without a toroidal array is refused, with its reason
    print(f"shot {SHOT}: {error}")

In [ ]:
midplane_names = [f"OutMirnov_{clock}_L1-03" for clock in (45, 135, 225)]
has_array = all(name in probe_names for name in midplane_names)
if has_array:
    midplane = [probe_names.index(name) for name in midplane_names]
    angles = sorted({round(float(np.degrees(ods[f"magnetics.b_field_pol_probe.{i}.position.phi"])), 1)
                     for i in midplane})
    spacing = np.diff(np.array(angles + [angles[0] + 360.0]))
    # Every n differing by 360 / gcd(spacings) predicts the same phases -- the gcd,
    # not the smallest spacing (probes at 0, 90 and 135 deg leave n modulo 8, not 4).
    step = math.gcd(360, *(int(round(value)) for value in spacing))
    modulus = 360 // step
    residues = range(1 - modulus // 2, modulus // 2 + 1)   # one n per residue class
    print("toroidal angles that recorded:", angles)
    print(f"spacings {spacing.round(1).tolist()} deg, gcd {step} deg -> n is resolved modulo {modulus}; "
          f"candidates {list(residues)}")
    try:
        vaft.omas.plot_mirnov_spatial_phase(ods, time=t_mode, channels=midplane, num_modes=2,
                                            candidate_n=range(0, 5), window_size=2000)
        plt.show()
    except ValueError as error:   # e.g. a dead channel in the trio: refused, with its reason
        print(f"the fit refuses: {error}")
else:
    modulus, residues = None, range(0)
    print(f"shot {SHOT} has no outboard fluctuation array: no toroidal mode number from this input")

Read the title before the number: it states how many *distinct toroidal positions* answered.
The legend carries a modulus only where an alias of the fitted $n$ is inside the candidate set —
it appears when the ambiguity is real, not as decoration. Three positions spaced 90 degrees apart
leave $n$ **modulo 4**.

The second band is the harmonic near 14 kHz. Its fit lands on $n = 4 \equiv 0$ modulo 4, while
a harmonic of an $n = 1$ structure would carry $n = 2$: either the harmonic's phases are poorly
defined in this window, or it is not a simple harmonic. Three probes cannot say which.

Three points and a line with two parameters leave one degree of freedom, so a small residual is
weak evidence on its own. The array has more than one trio, though — the same three toroidal
angles repeat at five heights and in two coil rows — so ask whether they agree.

In [ ]:
trio_fits = {}
rows = ("L1-01", "L1-03", "L1-05", "L2-01", "L2-03", "L2-05") if has_array and np.isfinite(f_mode) else ()
if rows:
    print(f"fit at {t_mode * 1e3:.2f} ms, {f_mode / 1e3:.1f} kHz (the tracked frequency), 1 ms window, "
          f"n in {list(residues)}")
    print(f"{'trio':8s} {'Z [m]':>6} {'n':>3} {'rms residual':>13}")
else:
    print(f"shot {SHOT}: no trio fit -- " + ("no tracked frequency" if has_array else "no toroidal array"))
for row in rows:
    names = [f"OutMirnov_{clock}_{row}" for clock in (45, 135, 225)]
    if not all(name in probe_names for name in names):
        continue
    trio = [probe_names.index(name) for name in names]
    signals = np.array([np.asarray(ods[f"magnetics.b_field_pol_probe.{i}.voltage.data"], dtype=float) for i in trio])
    phi = np.array([float(ods[f"magnetics.b_field_pol_probe.{i}.position.phi"]) for i in trio])
    trio_time = np.asarray(ods[f"magnetics.b_field_pol_probe.{trio[0]}.voltage.time"], dtype=float)
    fit = toroidal_phase_fit_at_time(trio_time, signals, phi, center_time=t_mode, window_size=2000,
                                     frequencies=[f_mode], num_modes=1, candidate_n=residues)
    trio_fits[row] = fit.modes[0]
    z = float(ods[f"magnetics.b_field_pol_probe.{trio[0]}.position.z"])
    print(f"{row:8s} {z:6.2f} {fit.modes[0].n:3d} {np.degrees(fit.modes[0].rms_error):10.1f} deg")

The trios do **not** agree. Only the midplane trio of the first row puts its three phases on an
$n$ line to within a few degrees; the others return an $n$ with a residual of 30–40 degrees — a
fit asked for a number returns one, whether or not any line passes through the points. A vote
across trios would be the wrong response. The residual is the evidence: it says which fit
describes its data, and the honest statement is "$n = 1$ modulo 4 on the midplane trio, not
reproduced by the off-midplane trios". Why the others disagree — poloidal structure mixing into
the toroidal phase at $|Z| = 0.4$ m, a coil-polarity or gain difference, a second mode — is left to
the exercise, and the #1005 method notebook is where channel calibration is taken apart.

#### 1.8 Available is not informative — and now VAFT refuses

Sessions 02 and 03 taught that a refused plot is often an underived quantity. Here is the converse
case, on `EQUILIBRIUM_SHOT`. 39915 falls in the 35521–44155 gap where no toroidal array recorded.
Its archive nevertheless published DAQ field 171 twice — as equilibrium probe `C2-05` and as a
`:phase_reference` entry at a different angle — so an earlier VAFT offered the toroidal fit, ran it,
and compared a signal with itself: a phase difference of exactly zero by construction, and an $n$
that was an identity (issues [#724](https://github.com/VEST-Tokamak/vaft/issues/724) and
[#825](https://github.com/VEST-Tokamak/vaft/issues/825)). Since that fix the mapper publishes the
channel once, and the fit counts only *distinct acquisitions* — a byte-identical waveform is one
channel whatever angle it declares. Ask `available_plots`, then try the call.

In [ ]:
rows = vaft.omas.available_plots(reference, available_only=False)
row = next(row for row in rows if row["name"] == "mirnov_spatial_phase")
print(f"shot {EQUILIBRIUM_SHOT}: mirnov_spatial_phase available? {row['available']}")
print(f"  reason: {row['reason']}")
print(f"  toroidal array: {toroidal_array_for_shot(EQUILIBRIUM_SHOT)['name']}")
try:
    vaft.omas.plot_mirnov_spatial_phase(reference, num_modes=1)
except ValueError as error:
    print(f"  the call refuses: {error}")

A plot that runs and returns a number is not automatically an answer worth having, and
`available_plots` only ever reported whether a call would *raise*. What changed is that the
condition the call checks is now the physical one — two distinct acquisitions at two toroidal
angles — rather than a neighbour of it (two entries that declare two angles). The refusal
carries its reason; read it.

### Step 2 — Soft X-ray: emissivity fluctuations

Soft X-ray chords see line-integrated emissivity, $\int \epsilon\, dl$, dominated by the hot core.
A rotating island or kink modulates it, so SXR looks at the same perturbation as the Mirnov
probes from inside the plasma rather than from the wall. The two arrays on 45531 view the plasma
from two toroidal angles (0 and 240 degrees in IMAS $\phi$): a vertical array of 20 chords and a
bottom array of 16 chords, each chord through a beryllium and an aluminium filter.

```text
SXR raw signal -> baseline / optional vacuum subtraction -> spectrogram -> MHD band
               -> compare chords -> phase / toroidal structure -> f_Mirnov <-> f_SXR
```

#### 2.1 Chords and raw signals

In [ ]:
if has_sxr:
    figure, axes = plt.subplots(1, 3, figsize=(17, 5), gridspec_kw={"width_ratios": [1, 1, 2]})
    vaft.omas.plot_soft_x_rays_geometry_lines_of_sight(ods, ax=axes[0])
    vaft.omas.plot_machine_geometry_topview(ods, ax=axes[1])
    vaft.omas.plot_soft_x_rays_time_power(ods, ax=axes[2])
    figure.tight_layout()
    plt.show()
    sxr_time = np.asarray(ods["soft_x_rays.channel.0.brightness.time"], dtype=float)
    sxr_data = np.vstack([np.ravel(ods[f"soft_x_rays.channel.{i}.brightness.data"]) for i in range(len(sxr_names))])
    sxr_rate = 1.0 / float(np.median(np.diff(sxr_time)))
    sxr_phi = np.array([np.degrees(float(ods[f"soft_x_rays.channel.{i}.line_of_sight.first_point.phi"]))
                        for i in range(len(sxr_names))])
    print(f"{len(sxr_names)} chords at {sxr_rate / 1e3:.1f} kHz, {sxr_time[0] * 1e3:.2f} - {sxr_time[-1] * 1e3:.2f} ms; "
          f"toroidal angles {sorted(set(np.round(sxr_phi).tolist()))} deg")
else:
    print(f"shot {SHOT} carries no soft_x_rays: the SXR steps below report nothing")

#### 2.2 Baseline, and the optional vacuum reference

The raw traces sit on offsets that differ by array — one of them near $-2$ V — so nothing is
comparable until each chord's quiet level is removed. `sxr_band_signals` takes the median of the
quiet tail after the plasma (`baseline_start`, here 5 ms after the plasma offset) and subtracts it.
A vacuum shot's PF pickup can additionally be removed with `sxr_subtract_vacuum_reference`; that
step is optional by design, needs a vacuum shot of the same campaign, and is never applied behind
your back.

In [ ]:
if has_sxr:
    baseline_start = window[1] + 5.0e-3
    sxr_bands = soft_x_rays.sxr_band_signals(
        sxr_time, sxr_data, baseline_start=baseline_start, bands={"mhd": (2.0e3, 2.0e4)},
        fs=sxr_rate, time_range=window,
    )
    shown = [sxr_names.index(name) for name in ("Vertical SXR Ch 10", "Bottom two-filter SXR Al Ch 16")]
    inside_sxr = (sxr_time >= window[0]) & (sxr_time <= window[1])
    figure, axes = plt.subplots(1, 3, figsize=(17, 4), sharex=True)
    for i in shown:
        axes[0].plot(sxr_time[inside_sxr] * 1e3, sxr_data[i, inside_sxr], lw=0.6, label=sxr_names[i])
        axes[1].plot(sxr_bands.time * 1e3, sxr_bands.raw[i], lw=0.6, label=sxr_names[i])
        axes[2].plot(sxr_bands.time * 1e3, sxr_bands.bands["mhd"][i], lw=0.6, label=sxr_names[i])
    for axis, title in zip(axes, ("raw", "baseline removed", "2-20 kHz band")):
        axis.set_title(title)
        axis.set_xlabel("Time [ms]")
        axis.grid(alpha=0.5)
    axes[0].set_ylabel("SXR signal [V]")
    axes[0].legend(fontsize="small")
    figure.tight_layout()
    plt.show()
    print("vacuum-reference subtraction: not applied -- this input carries no vacuum shot to subtract")

In [ ]:
if has_sxr:
    sxr_chord = sxr_names.index("Bottom two-filter SXR Al Ch 16")
    figure, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
    vaft.omas.plot_mirnov_spectrogram(ods, channel=mode_index, method="stft", time_range=window,
                                      nperseg=2048, max_frequency=3.0e4, track=(2.0e3, 2.0e4),
                                      max_jump=3.0e3, ax=axes[0])
    vaft.omas.plot_soft_x_rays_spectrogram(ods, channel=sxr_chord, time_range=window, max_frequency=3.0e4,
                                           track=(2.0e3, 2.0e4), max_jump=3.0e3, ax=axes[1])
    axes[0].set_xlabel("")
    plt.show()

#### 2.3 The same branch from inside

Both maps are drawn on the same window with the same tracker. The SXR chord shows the lower
branch strongly in the second half of the window, where the Mirnov track sits at 3–4 kHz; the
early 7 kHz line is weak in this chord. Whether "the same branch" is the same *mode* is a question
for coherence (step 5), not for the eye.

In [ ]:
if has_sxr:
    chord_rms = np.std(sxr_bands.bands["mhd"], axis=1)
    level = np.std(sxr_bands.raw, axis=1)
    figure, axes = plt.subplots(figsize=(12, 4))
    for array_name, marker in (("Vertical", "o"), ("Bottom two-filter SXR Be", "s"), ("Bottom two-filter SXR Al", "^")):
        members = [i for i, name in enumerate(sxr_names) if name.startswith(array_name)]
        chord = [int(sxr_names[i].split("Ch ")[-1]) for i in members]
        axes.plot(chord, chord_rms[members] / level[members], marker=marker, label=array_name)
    axes.set_xlabel("chord number")
    axes.set_ylabel("2-20 kHz rms / total rms")
    axes.set_title(f"#{SHOT}: relative MHD-band fluctuation per chord, {window[0] * 1e3:.1f}-{window[1] * 1e3:.1f} ms")
    axes.legend()
    axes.grid(alpha=0.5)
    plt.show()

#### 2.4 Comparing chords

Normalising the band rms by each chord's own rms removes the chords' different brightness and
gain, and leaves where along each array the fluctuation is a large *fraction* of the signal. A
chord is an integral, so this is a projection of structure, not a radial profile: a peak says
which lines of sight cross the fluctuating region, and inverting that into $\delta\epsilon(R, Z)$
is a tomography problem this session does not attempt.

In [ ]:
if has_sxr:
    pairs = soft_x_rays.sxr_te_pairs_from_ods(ods, "bottom")
    calibration, _te_table, _ratio_table = soft_x_rays.load_te_ratio_calibration()
    two_filter = soft_x_rays.sxr_electron_temperature(
        sxr_time, sxr_data, pairs, calibration=calibration, baseline_start=baseline_start,
        fs=sxr_rate, time_range=window,
    )
    print(f"{len(pairs)} Be/Al chord pairs on the bottom array")
    print("chord  median Te proxy in the plasma window [eV]")
    for (be, _al), te in zip(pairs, two_filter.te):
        print(f"{sxr_names[be].split('Ch ')[-1]:>5}  {np.nanmedian(te):8.1f}")

#### 2.5 A two-filter temperature proxy, labelled as one

The Be/Al ratio maps to $T_e$ through a packaged calibration, chord by chord. It is a
line-integrated, emission-weighted **proxy** — the calibration extrapolates rather than clips
outside its table, and a chord that grazes the edge reports a ratio, not a local temperature. Use
it for relative behaviour along the array and in time, not as a profile.

In [ ]:
if has_sxr and not np.isfinite(f_mode):
    print(f"shot {SHOT}: no tracked frequency -- no band to take SXR phases in")
elif has_sxr:
    narrow = soft_x_rays.sxr_band_signals(
        sxr_time, sxr_data, baseline_start=baseline_start,
        bands={"mode": (0.5 * f_mode, 1.5 * f_mode)}, fs=sxr_rate, time_range=window,
    )
    vertical = sxr_names.index("Vertical SXR Ch 10")
    bottom = sxr_names.index("Bottom two-filter SXR Al Ch 16")
    phase_v, envelope_v, *_ = soft_x_rays.hilbert_instantaneous_phase(narrow.bands["mode"][vertical], narrow.time, t_mode)
    phase_b, envelope_b, *_ = soft_x_rays.hilbert_instantaneous_phase(narrow.bands["mode"][bottom], narrow.time, t_mode)
    candidates = soft_x_rays.rank_toroidal_mode_numbers(sxr_phi[vertical], phase_v, sxr_phi[bottom], phase_b, n_max=3)
    print(f"at {t_mode * 1e3:.2f} ms, {0.5 * f_mode / 1e3:.1f}-{1.5 * f_mode / 1e3:.1f} kHz band:")
    print(f"  {sxr_names[vertical]:32s} phi {sxr_phi[vertical]:5.0f} deg  phase {phase_v:7.1f} deg")
    print(f"  {sxr_names[bottom]:32s} phi {sxr_phi[bottom]:5.0f} deg  phase {phase_b:7.1f} deg")
    print("  n candidates, best first (|residual| in deg):")
    for candidate in candidates:
        print(f"    n = {candidate.n:+d}   residual {abs(candidate.residual_deg):6.1f}")

#### 2.6 Phase, and why two chords do not give $n$

The Hilbert phase of a band-passed chord is its instantaneous phase; comparing two arrays 120
degrees apart ranks candidate $n$. Read the ranking honestly. Two positions 120 degrees apart
leave $n$ modulo 3, so candidates tie in pairs. Worse, the two chords do not view the same
poloidal location, so their phase difference contains $m\,\Delta\theta$ as well as
$n\,\Delta\phi$ — and $\Delta\theta$ of a line integral is not even one number. The ranking is a
list of what the two phases permit, not a measurement of $n$.

In [ ]:
if has_sxr:
    inside_mode = (mode_time >= window[0]) & (mode_time <= window[1])
    sxr_evolution = compute_spectrogram(sxr_time[inside_sxr], sxr_data[sxr_chord, inside_sxr],
                                        window_duration=1.0e-3, overlap=0.5)
    sxr_track = track_dominant_frequency(sxr_evolution, search_range=(2.0e3, 2.0e4), max_jump=3.0e3)
    mirnov_track = track_dominant_frequency(
        compute_spectrogram(mode_time[inside_mode], mode_voltage[inside_mode], window_duration=1.0e-3, overlap=0.5),
        search_range=(2.0e3, 2.0e4), max_jump=3.0e3)
    figure, axes = plt.subplots(figsize=(8, 4))
    axes.plot(mirnov_track.time * 1e3, mirnov_track.frequency / 1e3, "o-", label=f"Mirnov {mode_probe}")
    axes.plot(sxr_track.time * 1e3, sxr_track.frequency / 1e3, "s--", label=f"SXR {sxr_names[sxr_chord]}")
    axes.set_xlabel("Time [ms]")
    axes.set_ylabel("tracked frequency [kHz]")
    axes.legend()
    axes.grid(alpha=0.5)
    plt.show()
    both = np.isfinite(mirnov_track.frequency) & np.isfinite(np.interp(mirnov_track.time, sxr_track.time, sxr_track.frequency))
    agree = np.abs(mirnov_track.frequency - np.interp(mirnov_track.time, sxr_track.time, sxr_track.frequency)) <= 1.0e3
    print(f"windows tracked by both: {both.sum()}; within 1 kHz of each other: {(both & agree).sum()}")

#### 2.7 $f_{\mathrm{Mirnov}} \leftrightarrow f_{\mathrm{SXR}}$

Where both trackers report, their frequencies are compared window by window. They agree early and
late; in between, the probe's strongest line is near 9–10 kHz while the chord's is at 3–4 kHz. A
ridge tracker reports the *strongest* line of each record, and two diagnostics weight lines
differently — a magnetic probe at the wall favours what the chord barely sees. Agreement of the
ridge is necessary for "the same perturbation" and nowhere near sufficient: two unrelated signals
can share a frequency for a few milliseconds. Step 5 asks the sharper question.

### Step 3 — The fast camera: spatial fluctuation structure

The camera had a different job in each session: visible plasma evolution in 02, the equilibrium
overlay in 03, and here the **spatial structure of a fluctuation**. `CAMERA_SHOT = 40600` holds
602 frames at 50 kframe/s beside the outboard probe the published camera analyses use (DAQ field
171, `C2-05`), on one clock. The chain follows those papers, as implemented in
`vaft.process.camera_fluctuation`:

1. subtract each pixel's local temporal background;
2. take $f_{\mathrm{MHD}}(t)$ from the **magnetic probe** as the reference frequency;
3. form each pixel's short-time spectrum and integrate it about $f_{\mathrm{MHD}}(t)$;
4. divide by the local emission: $P_{\mathrm{camera}}(x_{\mathrm{pixel}}, y_{\mathrm{pixel}}, f_{\mathrm{MHD}})$.

Optical intensity is not density: visible emission on VEST is line radiation, so every map below
is an *emission* fluctuation measure.

#### 3.1 The probe sets the reference frequency

In [ ]:
camera_timing = plasma_timing(camera_ods)
camera_probe_names = [str(camera_ods[f"magnetics.b_field_pol_probe.{i}.name"])
                      for i in range(len(camera_ods["magnetics.b_field_pol_probe"]))]
camera_probe = camera_probe_names.index(eq_probe)
frame_root = "camera_visible.channel.0.detector.0.frame"
frame_times = np.array([float(camera_ods[f"{frame_root}.{k}.time"]) for k in range(len(camera_ods[frame_root]))])
camera_window = (float(frame_times[0]), float(frame_times[-1]))

probe_time = np.asarray(camera_ods[f"magnetics.b_field_pol_probe.{camera_probe}.voltage.time"], dtype=float)
probe_voltage = np.asarray(camera_ods[f"magnetics.b_field_pol_probe.{camera_probe}.voltage.data"], dtype=float)
probe_evolution = compute_spectrogram(probe_time, probe_voltage, window_duration=1.0e-3, overlap=0.5)
f_mhd = camera_fluctuation.track_reference_frequency(probe_evolution, search_range=(3.0e3, 1.5e4))
probe_track = track_dominant_frequency(probe_evolution, search_range=(3.0e3, 1.5e4))
in_camera = (probe_track.time >= camera_window[0]) & (probe_track.time <= camera_window[1])
t_camera = float(probe_track.time[in_camera][np.nanargmax(probe_track.power[in_camera])])

figure, axes = vaft.omas.plot_mirnov_spectrogram(
    camera_ods, channel=camera_probe, method="stft", time_range=camera_window,
    nperseg=256, max_frequency=2.5e4, track=(3.0e3, 1.5e4), max_jump=3.0e3,
)
axes.axvline(t_camera, color="k", ls="--", lw=0.8)
plt.show()
print(f"shot {CAMERA_SHOT}: plasma {camera_timing.onset * 1e3:.2f} - {camera_timing.offset * 1e3:.2f} ms, "
      f"camera {camera_window[0] * 1e3:.2f} - {camera_window[1] * 1e3:.2f} ms, {frame_times.size} frames")
print(f"strongest probe window inside the camera record: {t_camera * 1e3:.2f} ms")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 5))
vaft.omas.plot_camera_visible_image_frame(camera_ods, time=t_camera, ax=axes[0])
vaft.omas.plot_camera_visible_image_fluctuation(camera_ods, time=t_camera, ax=axes[1])
figure.tight_layout()
plt.show()

#### 3.2 Background subtraction

The raw frame is dominated by the slowly varying emission. Subtracting each pixel's mean over the
15 surrounding frames (0.3 ms) leaves what changed faster than that — the fluctuation, and noise.
The structure that survives is the input to the spectral step; on its own a single background-
subtracted frame cannot say whether the pattern is coherent.

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
vaft.omas.plot_mirnov_spectrogram(camera_ods, channel=camera_probe, method="stft", time_range=camera_window,
                                  nperseg=250, max_frequency=2.5e4, ax=axes[0])
vaft.omas.plot_camera_visible_spectrogram(camera_ods, nperseg=50, max_frequency=2.5e4, ax=axes[1])
axes[0].set_xlabel("")
plt.show()

#### 3.3 The same component in the light

The camera's whole-frame summed intensity, background removed, analysed with the same 1 ms
window as the probe. The camera's Nyquist frequency is 25 kHz, so its map stops there; the probe
map is cropped to match.

In [ ]:
frames = np.stack([np.asarray(camera_ods[f"{frame_root}.{k}.image_raw"], dtype=float) for k in range(frame_times.size)])
fluctuation_frames = camera_fluctuation.subtract_temporal_background(frames, window_frames=15)
pixels = camera_fluctuation.pixelwise_spectrogram(fluctuation_frames, frame_times, window_frames=50, overlap=0.5)
centre = np.interp(pixels.time, probe_evolution.time, f_mhd)
band = camera_fluctuation.mhd_band_power(pixels, centre_frequency=centre, half_width=500.0)
normalised = camera_fluctuation.normalize_by_local_emission(
    band, frames, frame_time=frame_times, power_time=pixels.time, window_frames=10)
step = int(np.argmin(np.abs(pixels.time - t_camera)))

figure, axes = plt.subplots(1, 2, figsize=(13, 5))
image = axes[0].imshow(normalised[step], cmap="turbo")
figure.colorbar(image, ax=axes[0], label="band power / local emission")
axes[0].set_title(f"#{CAMERA_SHOT} {pixels.time[step] * 1e3:.2f} ms: "
                  f"{centre[step] / 1e3:.1f} +/- 0.5 kHz from the probe")
vaft.omas.plot_camera_visible_image_mhd_power(camera_ods, time=float(pixels.time[step]),
                                              centre_frequency=float(centre[step]), ax=axes[1])
figure.tight_layout()
plt.show()
print(f"pixel spectrogram: {pixels.time.size} windows x {pixels.frequency.size} frequencies x "
      f"{pixels.pixel_shape[0]} x {pixels.pixel_shape[1]} pixels, df = {pixels.frequency[1] / 1e3:.1f} kHz")

#### 3.4 Where in the image the magnetic frequency lives

Left, the chain written out with the process functions over the whole record; right, the one-call
plot given the same centre frequency. The plot re-windows the record around the requested time,
so its transform windows do not fall where the full-record ones do — and with 1 kHz bins, a
$\pm 0.5$ kHz band holds a single bin, so exactly where the windows fall matters. Structure that
appears in both maps is the signal; what differs between them is the transform, the same rule as
step 1.4. The map is brightest where the emission fluctuates at the *probe's* frequency, relative
to how bright that part of the image is. This is how one perturbation appears in two diagnostics: a single number $f_{\mathrm{MHD}}$
from $\delta B$ at the wall, and a 2-D pattern in projected light.

Mind the projection: a camera pixel integrates along its line of sight through the whole plasma
and edge, so a bright region is where the *projection* fluctuates, not necessarily where the
perturbation is largest.

In [ ]:
camera_signal = camera_fluctuation.summed_region_signal(fluctuation_frames)
track_window = 2.0e-3
camera_evolution = compute_spectrogram(frame_times, camera_signal, window_duration=track_window, overlap=0.9)
probe_long = compute_spectrogram(probe_time, probe_voltage, window_duration=track_window, overlap=0.9)
camera_track = track_dominant_frequency(camera_evolution, search_range=(3.0e3, 1.5e4))
probe_long_track = track_dominant_frequency(probe_long, search_range=(3.0e3, 1.5e4))
probe_on_camera = np.interp(camera_track.time, probe_long_track.time, probe_long_track.frequency)
paired = np.isfinite(camera_track.frequency) & np.isfinite(probe_on_camera)
t_paired = camera_track.time[paired]
raw_r = stats.pearsonr(camera_track.frequency[paired], probe_on_camera[paired]).statistic
camera_residual = camera_track.frequency[paired] - np.polyval(np.polyfit(t_paired, camera_track.frequency[paired], 1), t_paired)
probe_residual = probe_on_camera[paired] - np.polyval(np.polyfit(t_paired, probe_on_camera[paired], 1), t_paired)
detrended_r = stats.pearsonr(camera_residual, probe_residual).statistic
# The windows overlap by 90 %: neighbours share most of their samples, so the
# paired windows are not independent and pearsonr's p-value (which assumes they
# are) does not apply.  Count the independent windows the span holds instead.
n_effective = (t_paired[-1] - t_paired[0]) / track_window

figure, axes = plt.subplots(figsize=(8, 4))
axes.plot(camera_track.time * 1e3, probe_on_camera / 1e3, "o-", label=f"probe {eq_probe}")
axes.plot(camera_track.time * 1e3, camera_track.frequency / 1e3, "s--", label="camera, whole frame")
axes.set_xlabel("Time [ms]")
axes.set_ylabel("tracked frequency [kHz]")
axes.set_title(f"#{CAMERA_SHOT}: 2 ms windows, 3-15 kHz search")
axes.legend()
axes.grid(alpha=0.5)
plt.show()
print(f"paired windows: {paired.sum()}, overlapping; independent windows in their "
      f"{(t_paired[-1] - t_paired[0]) * 1e3:.1f} ms span: N_eff ~ {n_effective:.0f}")
print(f"track correlation, raw      : r = {raw_r:+.2f}")
print(f"track correlation, detrended: r = {detrended_r:+.2f}  (no p-value: the windows overlap)")

#### 3.5 Does the camera follow the probe?

Two independent instruments, each tracked on its own. The *raw* correlation is inflated by any
common trend — both could simply drift down together — so the detrended residuals are the test:
do the wiggles on top of the trend agree? They do here — but count what is being correlated. The
windows overlap by 90 %, so neighbouring points share most of their samples, and the span holds
only the handful of independent windows printed as $N_{\mathrm{eff}}$. A p-value computed as if
all the points were independent would be wildly optimistic, so none is printed. With a handful of
independent windows a high $r$ is suggestive agreement, not a significance test. What the tracks actually show is worth saying plainly: both sit near 9–10
kHz, jump together to about 14 kHz, return near 11 kHz, and after about 315 ms the probe's line
falls to 5–6 kHz and then fades below the tracker's floor, while the camera's sits at 3–4 kHz
near the bottom of the search band. That is not a clean chirp, and the correlation depends on the
windows: the sample's manifest records $r = 0.67$ for 2 ms windows stepped by one frame — 503
windows, but only about five independent ones in their 10 ms span.

### Step 4 — Density and edge diagnostics

Treated briefly, because on these inputs they can say little about a coherent mode — and that is
itself the lesson.

- **Interferometer** — $\delta\int n_e\,dl$, line-integrated; a line-integrated density fluctuation
  is not a local density mode.
- **Triple Langmuir probe** — local edge $n_e(t)$, $T_e(t)$, processed at 25 kHz.
- **H-alpha filterscope** — line emission at 25 kHz: onset, envelope, slow modulation.

In [ ]:
if "Interferometer" not in bandwidths:
    print(f"shot {SHOT}: no interferometer record -- no line-integrated density fluctuation here")

figure, axes = plt.subplots(3, 1, figsize=(9, 9))
if "langmuir_probes" in ods:
    for i in range(len(ods["langmuir_probes.embedded"])):
        probe = ods[f"langmuir_probes.embedded.{i}"]
        lp_time = np.asarray(probe["time"], dtype=float)
        valid = np.asarray(probe["n_e.validity_timed"]) >= 0
        axes[0].plot(lp_time[valid] * 1e3, np.asarray(probe["n_e.data"])[valid], ".", ms=3, label=str(probe["name"]))
        axes[1].plot(lp_time[valid] * 1e3, np.asarray(probe["t_e.data"])[valid], ".", ms=3, label=str(probe["name"]))
        print(f"{probe['name']}: {valid.sum()} of {valid.size} samples valid, "
              f"{1.0 / float(np.median(np.diff(lp_time))) / 1e3:.0f} kHz")
    axes[0].set_yscale("log")
    axes[0].set_ylabel("n_e [m$^{-3}$]")
    axes[1].set_ylabel("T_e [eV]")
    axes[0].legend(fontsize="small")
else:
    print(f"shot {SHOT}: no Langmuir probe record")
if "spectrometer_uv" in ods:
    vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="H_alpha", ax=axes[2])
    axes[2].set_xlim(floor_window[0], window[1] + 5.0e-3)
else:
    print(f"shot {SHOT}: no spectrometer_uv record -- no filterscope")
for axis in axes[:2]:
    axis.set_xlim(floor_window[0] * 1e3, (window[1] + 5.0e-3) * 1e3)
    axis.grid(alpha=0.5)
figure.tight_layout()
plt.show()

# Labels carry a rate suffix when one diagnostic was stored at several rates, so
# the filterscope is found by its prefix; with several rates the slowest bounds it.
filterscope_nyquists = [nyquist for label, (_rate, nyquist) in bandwidths.items()
                        if label.startswith("Filterscope / UV line")]
if not filterscope_nyquists:
    print(f"shot {SHOT}: no filterscope record -- nothing to bound")
elif not tracked.any():
    print(f"H-alpha Nyquist {min(filterscope_nyquists) / 1e3:.1f} kHz; no tracked line to compare it with")
else:
    h_alpha_nyquist = min(filterscope_nyquists)
    print(f"H-alpha Nyquist {h_alpha_nyquist / 1e3:.1f} kHz; the tracked line at "
          f"{np.nanmin(mode_track.frequency) / 1e3:.0f}-{np.nanmax(mode_track.frequency) / 1e3:.0f} kHz gets "
          f"{2.0 * h_alpha_nyquist / np.nanmax(mode_track.frequency):.1f}-{2.0 * h_alpha_nyquist / np.nanmin(mode_track.frequency):.1f} "
          f"samples per period")

The Langmuir probes carry validity flags on every sample, and only the samples the mapper marked
valid are drawn — most of the record is flagged. At 25 kHz, with Nyquist 12.5 kHz, both the probes
and the filterscope sample the 3–9 kHz line with only three to eight samples per period, and cannot
represent the 14 kHz harmonic at all. They can corroborate the *envelope* and the *timing* of activity; they
cannot corroborate a mode. **The same event timing is not the same coherent mode.**

### Step 5 — Several diagnostics at once: coincidence and coherence

This is the integrated heart of the session:

$$
\delta B \;\leftrightarrow\; \delta I_{\mathrm{SXR}} \;\leftrightarrow\; \delta I_{\mathrm{camera}}
\;\leftrightarrow\; \delta n_e \;\leftrightarrow\; \delta I_{H\alpha}.
$$

#### 5.1 Coincidence first

Normalised envelopes on one axis: when does each diagnostic become active?

In [ ]:
figure, axes = plt.subplots(figsize=(9, 4))
axes.plot(mode_evolution.time * 1e3, band_power / np.nanmax(band_power), label=f"Mirnov {mode_probe}, 2-40 kHz")
if has_sxr:
    sxr_envelope = compute_spectrogram(sxr_time, sxr_data[sxr_chord], window_duration=1.0e-3, overlap=0.5)
    sxr_power = np.array([compute_band_power(sxr_envelope.frequency, column, {"mhd": (2.0e3, 2.0e4)})["mhd"]
                          for column in sxr_envelope.magnitude.T])
    axes.plot(sxr_envelope.time * 1e3, sxr_power / np.nanmax(sxr_power), label=f"SXR {sxr_names[sxr_chord]}, 2-20 kHz")
# The H-alpha line by its stored label, not by position: channel 0 is H-alpha on
# 45531 but nothing guarantees it on another shot.
h_alpha_lines = ([f"spectrometer_uv.channel.{i}.processed_line.{j}"
                  for i in range(len(ods["spectrometer_uv.channel"]))
                  for j in range(len(ods[f"spectrometer_uv.channel.{i}.processed_line"]))
                  if "label" in ods[f"spectrometer_uv.channel.{i}.processed_line.{j}"]
                  and str(ods[f"spectrometer_uv.channel.{i}.processed_line.{j}.label"]).startswith("H-alpha")]
                 if "spectrometer_uv" in ods else [])
if h_alpha_lines:
    h_alpha_time = np.asarray(ods["spectrometer_uv.time"], dtype=float)
    h_alpha = np.asarray(ods[f"{h_alpha_lines[0]}.intensity.data"], dtype=float)
    axes.plot(h_alpha_time * 1e3, h_alpha / np.nanmax(h_alpha), label="H-alpha")
else:
    h_alpha_time = h_alpha = None
    print(f"shot {SHOT}: no H-alpha line in spectrometer_uv")
axes.plot(ip_time * 1e3, np.abs(ip) / np.abs(ip[i_peak]), "k", lw=0.8, label="|Ip|")
axes.set_xlim(floor_window[0] * 1e3, (window[1] + 5.0e-3) * 1e3)
axes.set_xlabel("Time [ms]")
axes.set_ylabel("normalised")
axes.legend(fontsize="small")
axes.grid(alpha=0.5)
plt.show()

Everything becomes active inside the plasma window, as it must — which is exactly why this
figure proves little. Two diagnostics growing at the same time are consistent with one
perturbation, with two, and with a discharge that simply got hotter. The sharper test is
spectral:

$$
S_{xy}(f), \qquad
C_{xy}(f) = \frac{|S_{xy}(f)|^2}{S_{xx}(f)\,S_{yy}(f)}, \qquad
\phi_{xy}(f) = \arg S_{xy}(f).
$$

Coherence near one at a frequency says the two signals keep a *fixed phase relation* there, window
after window. It must be read against its 95 % significance line, which depends only on how many
segments were averaged: with $N$ segments, independent noise exceeds $1 - 0.05^{1/(N-1)}$ one time
in twenty.

#### 5.2 Which chords are coherent with the probe

In [ ]:
if has_sxr:
    # The plasma window, and a control: the SXR record's start to the plasma onset. The
    # coils are energised inside the control (the OH coil fires before the plasma), so
    # coherence produced by electrical pickup alone would show up there too.
    intervals = {"plasma window": window, "pre-plasma control": (max(sxr_time[0], mode_time[0]), window[0])}
    scans, chords_above = {}, {}
    figure, axes = plt.subplots(figsize=(12, 4))
    for offset, (label, (start, stop)) in enumerate(intervals.items()):
        pick_mode = (mode_time >= start) & (mode_time <= stop)
        pick_sxr = (sxr_time >= start) & (sxr_time <= stop)
        results = [cross_spectrum(mode_time[pick_mode], mode_voltage[pick_mode],
                                  sxr_time[pick_sxr], sxr_data[i, pick_sxr], nperseg=2048)
                   for i in range(len(sxr_names))]
        in_band = (results[0].frequency >= 2.0e3) & (results[0].frequency <= 5.0e3)
        n_bins, n_segments = int(in_band.sum()), results[0].n_segments
        # Each chord is judged by its best of n_bins bins, so the per-bin 5 % level is
        # Bonferroni-corrected to 5 %/n_bins: noise then clears it in one chord in twenty.
        bonferroni = 1.0 - (0.05 / n_bins) ** (1.0 / (n_segments - 1))
        coherence = np.array([result.coherence[in_band] for result in results])
        chords_above[label] = int((coherence.max(axis=1) > bonferroni).sum())
        scans[label] = results
        bar_width = 1.0 / (len(intervals) + 1)
        bars = axes.bar(np.arange(len(sxr_names)) + bar_width * offset, coherence.max(axis=1), width=bar_width,
                        label=label)
        axes.axhline(bonferroni, color=bars.patches[0].get_facecolor(), ls="--", lw=1.0)
        print(f"{label} {start * 1e3:.1f}-{stop * 1e3:.1f} ms: {n_segments} segments, {n_bins} bins in "
              f"{results[0].frequency[in_band][0] / 1e3:.1f}-{results[0].frequency[in_band][-1] / 1e3:.1f} kHz; "
              f"single-bin 95 % line {results[0].significance_95:.2f}, Bonferroni line {bonferroni:.2f}")
        print(f"  chords above the line: {chords_above[label]} of {len(sxr_names)} (best bin above the Bonferroni "
              f"line; ~{0.05 * len(sxr_names):.1f} expected from noise)")
        print(f"  against the single-bin line instead: best bin above it in "
              f"{int((coherence.max(axis=1) > results[0].significance_95).sum())} chords "
              f"(~{(1.0 - 0.95 ** n_bins) * len(sxr_names):.0f} expected from noise), any bin above it in "
              f"{int((coherence > results[0].significance_95).sum())} of {coherence.size} bins (5 % expected)")
    scan = scans["plasma window"]
    low_branch = (scan[0].frequency >= 2.0e3) & (scan[0].frequency <= 5.0e3)
    peak_coherence = np.array([result.coherence[low_branch].max() for result in scan])
    axes.set_xticks(np.arange(len(sxr_names))[::4])
    axes.set_xticklabels([sxr_names[i].replace("two-filter SXR ", "") for i in range(0, len(sxr_names), 4)],
                         rotation=45, ha="right", fontsize="small")
    axes.set_ylabel("max coherence, 2-5 kHz\n(dashed: Bonferroni 95 % line)")
    axes.set_title(f"#{SHOT}: {mode_probe} against every SXR chord, plasma window and pre-plasma control")
    axes.legend()
    plt.show()
    best = int(np.argmax(peak_coherence))
    print(f"common grid: {scan[0].sample_rate / 1e3:.1f} kHz, resampled {scan[0].resampled}; "
          f"most coherent in the plasma window: {sxr_names[best]} ({peak_coherence[best]:.2f})")

Three cautions before using the most coherent chord.

**A maximum over bins is judged against a line for a maximum.** The printed 95 % line is for *one*
frequency bin; each chord's bar is its best of several. Against the single-bin line noise alone
would put about a quarter of the chords above it, $1 - 0.95^{N_{\mathrm{bins}}}$, and before the plasma
most of them clear it — so each bar is compared with the Bonferroni line $1 - (0.05/N_{\mathrm{bins}})^{1/(N-1)}$, which noise clears in one chord in
twenty. Across 52 chords that still leaves two or three chance passes, so a single chord just above
the line is weak; many chords well above it is not.

**A control says what pickup alone does.** Before the plasma the coils are already energised, so
electrical pickup common to the probe and the SXR amplifiers would be coherent without a plasma. In the pre-plasma control no more chords clear the Bonferroni line than
noise predicts, although the single-bin count sits above its 5 % — some shared pickup exists, and
it is weak. The plasma-window count is far above both. The control has fewer segments, and its line
is higher to say so; it is a check on pickup, not a matched null.

**Coherence is only half the answer** — the phase is the other half.

In [ ]:
if has_sxr:
    vaft.omas.plot_diagnostics_spectrum_coherence(
        ods, x_signal=f"mirnov:{mode_probe}", y_signal=f"sxr:{sxr_names[best]}",
        time_range=window, nperseg=2048, max_frequency=3.0e4,
    )
    plt.show()
    coherent = scan[best].coherence > scan[best].significance_95
    shown_bins = coherent & (scan[best].frequency <= 2.0e4)
    print(f"bins above the 95 % line below 20 kHz: "
          + ", ".join(f"{f / 1e3:.1f} kHz ({np.degrees(p):+.0f} deg)"
                      for f, p in zip(scan[best].frequency[shown_bins], scan[best].phase[shown_bins])))

#### 5.3 Coherence and phase, Mirnov against SXR

The title records the grid the two records were compared on and which was resampled — the 2 MHz
probe and the 976.6 kHz digitiser do not share a clock, and the overlap cut is never silent. Where
the coherence clears its line, the phase is marked: across the low branch it stays nearly constant,
a *fixed* phase between $\delta B$ at the wall and $\delta\epsilon$ along the chord. With the control of 5.2 —
the same comparison before the plasma clears the corrected line in no more chords than noise
would — that is evidence that both diagnostics respond to a common fluctuation carried by the
plasma, not to pickup, and it is much stronger than 5.1. It is consistent with one perturbation;
it does not by itself show that the coherent activity across the band is a single mode.

#### 5.4 How few segments fool you

Shorten the window and the same calculation averages over fewer segments. Watch the line.

In [ ]:
if has_sxr:
    short = (window[0], window[0] + 4.0e-3)
    pick_mode = (mode_time >= short[0]) & (mode_time <= short[1])
    pick_sxr = (sxr_time >= short[0]) & (sxr_time <= short[1])
    few = cross_spectrum(mode_time[pick_mode], mode_voltage[pick_mode], sxr_time[pick_sxr],
                         sxr_data[best, pick_sxr], nperseg=2048)
    for label, result in (("whole window", scan[best]), ("first 4 ms", few)):
        in_band = (result.frequency >= 2.0e3) & (result.frequency <= 2.0e4)
        print(f"{label:12s}: {result.n_segments:2d} segments, 95 % line {result.significance_95:.2f}, "
              f"coherence 2-20 kHz median {np.median(result.coherence[in_band]):.2f}, "
              f"max {result.coherence[in_band].max():.2f}")

With two segments the estimate has almost no averaging: pure noise then scatters over the whole
range from 0 to 1, and the 95 % line rises to 0.95 to say so. A coherence of 0.8 there says
nothing, while 0.8 over the whole window with twelve segments is strong. A coherence value
without its segment count is not a result.

#### 5.5 Probe against camera

In [ ]:
in_frames = (probe_time >= camera_window[0]) & (probe_time <= camera_window[1])
probe_camera = cross_spectrum(probe_time[in_frames], probe_voltage[in_frames], frame_times, camera_signal, nperseg=100)
vaft.plot.plot_cross_spectrum(probe_camera, x_label=f"probe {eq_probe}", y_label="camera, whole frame",
                              max_frequency=2.5e4,
                              title=f"#{CAMERA_SHOT}: camera relative to probe ({probe_camera.n_segments} segments, "
                                    f"{probe_camera.sample_rate / 1e3:.0f} kHz grid)")
plt.show()
above = probe_camera.coherence > probe_camera.significance_95
print(f"95 % line {probe_camera.significance_95:.2f}; coherent bins 3-15 kHz: "
      + ", ".join(f"{f / 1e3:.1f}" for f in probe_camera.frequency[above & (probe_camera.frequency >= 3.0e3)
                                                                  & (probe_camera.frequency <= 1.5e4)]) + " kHz")

The probe is resampled onto the camera's 50 kHz frame clock — the slower record sets the grid, and
the result says so. Bins near the tracked 9–10 and 14 kHz lines clear the line: the whole-frame
light and $\delta B$ at the wall keep a fixed phase there. Timing alone (step 3's two
spectrograms looking alike) would not have established that.

### Step 6 — From frequency to mode identification

Collect what 45531 has actually supported, each item with the step that measured it:

$$
\boxed{\text{frequency} + \text{spatial phase} + \text{mode number} + \text{multi-diagnostic consistency}}
$$

In [ ]:
evidence = [
    ("frequency f(t)", f"{np.nanmax(mode_track.frequency) / 1e3:.0f} kHz falling to "
                       f"{np.nanmin(mode_track.frequency) / 1e3:.0f} kHz", "1.3"),
    ("toroidal n", f"n = {trio_fits['L1-03'].n} modulo {array['alias_step']} on the midplane trio "
                   f"(rms {np.degrees(trio_fits['L1-03'].rms_error):.0f} deg); off-midplane trios disagree"
     if has_array and "L1-03" in trio_fits else "not measurable on this input", "1.7"),
    ("poloidal m", "not measured (no poloidal array analysis in VAFT; see below)", "--"),
    ("SXR coherence", f"{chords_above['plasma window']} of {len(sxr_names)} chords above the corrected line "
                      f"at 2-5 kHz ({chords_above['pre-plasma control']} before the plasma)"
     if has_sxr else "no SXR", "5.2"),
    ("camera coherence", f"{CAMERA_SHOT}, a different shot: coherent bins at the probe's lines", "5.5"),
    ("plasma rotation", "not measured on this shot", "7.1"),
]
for quantity, value, where in evidence:
    print(f"{quantity:18s} {value:70s} step {where}")

That is a coherent, rotating, low-$n$ magnetic perturbation seen by $\delta B$ and by core
emissivity with a fixed phase between them, whose frequency changes during the current flat-top.
It is not yet a named mode, and the missing entry is the reason. **Poloidal $m$ is not supported**:
the outboard probes sit outside the LCFS and, mapped onto the straight-field-line angle of
39915's equilibrium, span only about 16–20 degrees of $\theta^*$ — far too little to count
poloidal wavelengths, and VAFT has no flux-coordinate transform to do the mapping properly
(issue #472) nor a resolver from $(m, n)$ to a surface (issue
[#506](https://github.com/VEST-Tokamak/vaft/issues/506)). A spectrogram overlay of the
$f = n f_\phi$ track at each rational surface is issue
[#460](https://github.com/VEST-Tokamak/vaft/issues/460).

### Step 7 — Equilibrium and rotation: the physical interpretation

$$
\omega_{\mathrm{plasma}} = \omega_{\mathrm{lab}} - \mathbf{k}\cdot\mathbf{V}
\;\sim\; \omega_{\mathrm{lab}} - n\,\Omega_\phi + m\,\Omega_\theta,
\qquad q(\rho_s) = \frac{m}{n}.
$$

#### 7.1 How large is $n f_\phi$ on VEST?

45531 has no rotation measurement. `ROTATION_SHOT = 48224` has charge-exchange $v_\phi$ — a
**different discharge**, so what follows is an order-of-magnitude estimate of the Doppler term on
VEST, not a correction of 45531's frequency.

In [ ]:
cx_r = np.array([np.ravel(rotation[f"charge_exchange.channel.{c}.position.r.data"])[0]
                 for c in range(len(rotation["charge_exchange.channel"]))])
v_phi = np.array([np.ravel(rotation[f"charge_exchange.channel.{c}.ion.0.velocity_tor.data"])
                  for c in range(len(rotation["charge_exchange.channel"]))])
f_phi = np.abs(v_phi) / (2.0 * np.pi * cx_r[:, None])
print(f"shot {ROTATION_SHOT}: {len(cx_r)} CES channels, R = {cx_r.min():.2f} - {cx_r.max():.2f} m")
print(f"|v_phi| = {np.abs(v_phi).min() / 1e3:.0f} - {np.abs(v_phi).max() / 1e3:.0f} km/s  ->  "
      f"f_phi = {f_phi.min() / 1e3:.1f} - {f_phi.max() / 1e3:.1f} kHz")
for n in (1, 2):
    print(f"n = {n}: n f_phi = {n * f_phi.min() / 1e3:.0f} - {n * f_phi.max() / 1e3:.0f} kHz   "
          f"(shot {SHOT}'s tracked line: {np.nanmin(mode_track.frequency) / 1e3:.0f}-{np.nanmax(mode_track.frequency) / 1e3:.0f} kHz)")

On VEST the Doppler term for $n = 1$ is **as large as the observed line or larger**. So a 3–9 kHz
line cannot be read as an intrinsic mode frequency: it is compatible with a mode nearly at rest in
the plasma frame, carried round by rotation, and equally with a slower plasma and a mode with a
frequency of its own.
Deciding needs $v_\phi$ at the mode's surface in the *same* discharge — and the surface needs $m$.

#### 7.2 Which surface? $n$ alone does not say

`EQUILIBRIUM_SHOT` carries EFIT slices. For $n = 1$, every integer $m$ that $q$ passes through
has a surface.

In [ ]:
eq_times = np.asarray(reference["equilibrium.time"], dtype=float)
eq_ip = np.array([float(reference[f"equilibrium.time_slice.{i}.global_quantities.ip"]) for i in range(eq_times.size)])
usable = np.flatnonzero(eq_ip != 0.0)
i_rep = int(usable[np.argmax(np.abs(eq_ip[usable]))])
profiles = reference[f"equilibrium.time_slice.{i_rep}.profiles_1d"]
quantities = reference[f"equilibrium.time_slice.{i_rep}.global_quantities"]
psi_n = (np.asarray(profiles["psi"]) - float(quantities["psi_axis"])) / (
    float(quantities["psi_boundary"]) - float(quantities["psi_axis"]))
q = np.abs(np.asarray(profiles["q"], dtype=float))

figure, axes = plt.subplots(figsize=(8, 4.5))
axes.plot(psi_n, q, "k")
print(f"shot {EQUILIBRIUM_SHOT} at {eq_times[i_rep] * 1e3:.1f} ms: q0 = {q[0]:.2f}, q_edge = {q[-1]:.2f}")
for n, style in ((2, dict(marker="s", ms=4)), (1, dict(marker="o", ms=10, mfc="none"))):
    surfaces = vaft.process.equilibrium.find_rational_surfaces(psi_n, q, n=n)
    axes.plot(surfaces["psi_n_rational"], surfaces["q_rational"], ls="none", label=f"n = {n}", **style)
    print(f"n = {n}: {len(surfaces['m'])} surfaces, m = {surfaces['m'].min()}-{surfaces['m'].max()}, "
          f"psi_N = " + ", ".join(f"{p:.2f}" for p in surfaces["psi_n_rational"][:5]) + " ...")
axes.set_xlabel(r"$\psi_N$")
axes.set_ylabel("|q|")
axes.set_title(f"#{EQUILIBRIUM_SHOT} (not {SHOT}): rational surfaces q = m/n")
axes.legend()
axes.grid(alpha=0.5)
plt.show()

In [ ]:
tracked_q = (3.0, 4.0, 5.0)
surfaces_in_time = {value: [] for value in tracked_q}
print(f"{'t [ms]':>7} {'q0':>6} {'q95':>6} " + " ".join(f"{'q=' + str(int(value)):>7}" for value in tracked_q))
for i in usable:
    descriptors = vaft.process.equilibrium.derive_global_descriptors(
        vaft.process.equilibrium.as_equilibrium(reference, time_index=i), rational_q=(1.0, 2.0) + tracked_q)
    row = []
    for value in tracked_q:
        found = descriptors.rational_surfaces[value]
        surfaces_in_time[value].append(float(found[0].value) if found else np.nan)
        row.append(f"{surfaces_in_time[value][-1]:7.3f}" if found else f"{'--':>7}")
    print(f"{eq_times[i] * 1e3:7.1f} {descriptors['q0'].value:6.2f} {descriptors['q95'].value:6.2f} " + " ".join(row))
print(f"q = 1 surface at the representative slice: "
      f"{bool(vaft.process.equilibrium.derive_global_descriptors(vaft.process.equilibrium.as_equilibrium(reference, time_index=i_rep), rational_q=(1.0,)).rational_surfaces[1.0])}")

figure, axes = plt.subplots(figsize=(8, 4))
for value in tracked_q:
    axes.plot(eq_times[usable] * 1e3, surfaces_in_time[value], marker="o", label=f"q = {value:.0f}")
axes.set_xlabel("Time [ms]")
axes.set_ylabel(r"$\psi_N$ of the surface")
axes.set_ylim(0.0, 1.0)
axes.set_title(f"#{EQUILIBRIUM_SHOT}")
axes.legend()
axes.grid(alpha=0.5)
plt.show()

The profile has a surface for $m = 3, 4, 5, \dots$ at $n = 1$, and $q$ on axis is above two, so
there is no $q = 1$ or $q = 2$ surface: the familiar $1/1$ and $2/1$ readings are unavailable as a
*fact about this equilibrium*. Where $q = 3$ exists it sweeps a third of the minor radius in ten
milliseconds. So **measuring $n$ alone does not determine a resonant surface** — the candidates
differ by which $m$ one assumes, and the assumption is exactly what is not measured.

### Step 8 — Transient events as sequences

An IRE or a disruption is not a frequency; it is an ordered sequence of measured changes. The
tools return numbers:

- `current_quench(time, ip)` — reference current and time, the 80 % and 20 % crossings, the 80–20
  duration and its extrapolation to 100 %, and the steepest $dI_p/dt$;
- `current_spike(time, ip, before=...)` — the largest positive excursion of $|I_p|$ above its
  trend, accepted only above 5 robust sigmas; otherwise a `reason`;
- `vertical_position_history(ods)` — the magnetic axis $Z(t)$ and $dZ/dt$ over the equilibrium
  slices, matched by time.

Here they are on `SHOT`, with the fluctuation power and the light on the same axis.

In [ ]:
quench = current_quench(ip_time, ip)
# A spike is searched inside the quench; with no quench there is nothing to search.
spike = (current_spike(ip_time, ip, before=quench.time_20, lookback_s=quench.duration_80_20)
         if quench.found else None)
vertical = vaft.omas.vertical_position_history(ods)

figure, axes = plt.subplots(4, 1, figsize=(9, 10), sharex=True)
axes[0].plot(ip_time * 1e3, np.abs(ip) / 1e3, "k")
axes[0].set_ylabel("|Ip| [kA]")
axes[1].semilogy(mode_evolution.time * 1e3, band_power)
axes[1].set_ylabel(f"Mirnov 2-40 kHz\n[V$^2$]")
if has_sxr:
    axes[2].semilogy(sxr_envelope.time * 1e3, sxr_power)
axes[2].set_ylabel("SXR 2-20 kHz")
if h_alpha is not None:
    axes[3].plot(h_alpha_time * 1e3, h_alpha)
axes[3].set_ylabel("H-alpha [a.u.]")
axes[3].set_xlabel("Time [ms]")
marks = {"80 %": quench.time_80, "20 %": quench.time_20, "min dIp/dt": quench.time_didt_min}
if spike is not None and spike.time is not None:
    marks["spike"] = spike.time
for axis in axes:
    axis.set_xlim(floor_window[0] * 1e3, (window[1] + 5.0e-3) * 1e3)
    axis.grid(alpha=0.5)
    for label, when in marks.items():
        if when is not None:
            axis.axvline(when * 1e3, ls="--", lw=0.8, color="r" if label == "spike" else "0.4")
axes[0].set_title(f"#{SHOT}: dashed = quench 80 % / 20 % / steepest fall" + (", red = spike" if "spike" in marks else ""))
plt.show()

if quench.found:
    print(f"reference current {quench.reference_current / 1e3:.1f} kA at {quench.reference_time * 1e3:.2f} ms")
    print(f"80 % at {quench.time_80 * 1e3:.2f} ms, 20 % at {quench.time_20 * 1e3:.2f} ms: "
          f"80-20 time {quench.duration_80_20 * 1e3:.2f} ms")
    print(f"steepest dIp/dt {quench.didt_min / 1e6:.1f} MA/s at {quench.time_didt_min * 1e3:.2f} ms")
    print(f"spike inside the quench: " + (f"+{spike.amplitude / 1e3:.1f} kA at {spike.time * 1e3:.2f} ms"
                                          if spike.time is not None else spike.reason))
else:
    print(f"current quench: {quench.reason}; no spike search without one")
print(f"vertical position: {vertical.reason or 'found'}")

Read the panels as a sequence, top to bottom and left to right: the coherent magnetic line of
steps 1–5 is present through the flat-top; the SXR band grows; the current then falls from 80 % to
20 % in the printed time; the fluctuation power and the light collapse with it. That is a
precursor, a quench and a termination, in order, with numbers. Whether it is an IRE, a minor or a
major disruption is not something any of these functions decides, and there is no classifier in
VAFT that would — so nothing here is named.

`vertical_position_history` needs equilibrium slices, and 45531 has none: it says so rather than
returning zeros. The same tools on `EQUILIBRIUM_SHOT` show two further lessons.

In [ ]:
ref_time = np.asarray(reference["magnetics.ip.0.time"], dtype=float)
ref_ip = np.asarray(reference["magnetics.ip.0.data"], dtype=float)
print(f"shot {EQUILIBRIUM_SHOT}:")
for smoothing in (None, 2.0e-4, 5.0e-4):
    ref_quench = current_quench(ref_time, ref_ip, smoothing_s=smoothing)
    label = "raw" if smoothing is None else f"{smoothing * 1e3:.1f} ms smoothing"
    print(f"  {label:18s}: 80-20 time {ref_quench.duration_80_20 * 1e3:.2f} ms, "
          f"steepest dIp/dt {ref_quench.didt_min / 1e6:6.1f} MA/s at {ref_quench.time_didt_min * 1e3:.2f} ms"
          if ref_quench.found else f"  {label:18s}: {ref_quench.reason}")
ref_spike = (current_spike(ref_time, ref_ip, before=ref_quench.time_20, lookback_s=ref_quench.duration_80_20)
             if ref_quench.found else None)
if ref_spike is None:
    print(f"  current quench: {ref_quench.reason}")
else:
    print(f"  spike inside the quench: +{ref_spike.amplitude / 1e3:.1f} kA at {ref_spike.time * 1e3:.2f} ms"
          if ref_spike.time is not None else f"  spike: {ref_spike.reason}")

ref_vertical = vaft.omas.vertical_position_history(reference)
ref_timing = plasma_timing(reference)
figure, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(ref_time * 1e3, np.abs(ref_ip) / 1e3, "k")
if ref_quench.found:
    axes[0].axvspan(ref_quench.time_80 * 1e3, ref_quench.time_20 * 1e3, alpha=0.15, label="80-20 %")
if ref_spike is not None and ref_spike.time is not None:
    axes[0].axvline(ref_spike.time * 1e3, color="r", ls="--", lw=0.8, label="spike")
axes[0].set_ylabel("|Ip| [kA]")
axes[0].legend()
axes[1].plot(ref_vertical.time * 1e3, ref_vertical.z_axis * 1e3, "o-")
axes[1].set_ylabel("Z_axis [mm]")
axes[1].set_xlabel("Time [ms]")
for axis in axes:
    axis.set_xlim((ref_timing.onset - 5.0e-3) * 1e3, (ref_timing.offset + 5.0e-3) * 1e3)
    axis.grid(alpha=0.5)
axes[0].set_title(f"#{EQUILIBRIUM_SHOT}")
plt.show()
jump = int(np.nanargmax(np.abs(ref_vertical.dz_dt)))
z_valid = ref_vertical.z_axis[ref_vertical.valid]
print(f"largest |dZ/dt|: {ref_vertical.dz_dt[jump]:+.1f} m/s at {ref_vertical.time[jump] * 1e3:.1f} ms; "
      f"Z_axis values: {sorted(set(np.round(z_valid * 1e3, 1).tolist()))} mm "
      f"(smallest |Z_axis| {np.min(np.abs(z_valid)) * 1e6:.2f} um)")

Two lessons.

**A steepest slope depends on how it was taken.** The 80–20 time is identical in all three rows
by construction, not because it proved robust: `current_quench` takes the crossings from the raw
trace, and `smoothing_s` only changes how the *rate* is taken. The rate is what moves — the
steepest $dI_p/dt$ changes by more than a factor of two, because a positive current spike sits
*inside* the quench and the derivative of raw data rings on it. Quote the smoothing with the rate,
or quote the 80–20 time, whose crossings are read from the raw trace by design.

**A derivative of a reconstruction inherits its artefacts.** $Z_{\mathrm{axis}}$ takes only two
values — about 23 mm, then ~0 (sub-micron) from one slice to the next — so the "vertical velocity"
between them is a jump between two reconstructions, most likely a change in how EFIT was
constrained, not a plasma moving. A vertical displacement event is a *measured* motion; a step in
a fitted parameter is not one.

## Interpretation Checkpoints

Each has a definite answer in what you have already computed.

1. **The frequency on 45531 falls from about 7 to about 3 kHz.** Name two physically different
   explanations using $f_{\mathrm{lab}} \approx f_{\mathrm{plasma}} + n f_\phi$. Which
   measurement would separate them, and on which shot of this session does it exist?
2. **The midplane trio gives $n = 1$ modulo 4 with a small residual; the other trios do not.**
   What does the residual tell you that the number does not? Would several off-midplane trios
   agreeing on one other $n$, each with a 30–40 degree residual, be stronger evidence?
3. **39915 now refuses the toroidal fit.** It used to run and return a number. What was wrong
   with that number, and why is "the call raised" the better outcome?
4. **In a 4 ms window a chord reaches a high coherence with the probe.** Why is that not evidence,
   and what single number printed beside it tells you so?
5. **The camera tracks the probe with a detrended correlation well above zero.** Why is the
   detrended value the one to report, why is a raw correlation of two falling tracks weak, and
   why is no p-value printed beside 36 windows that overlap by 90 %?
6. **H-alpha rises when the Mirnov band power does.** Can the filterscope confirm the 14 kHz
   harmonic? Can it confirm the timing?
7. **39915 has $n = 1$ surfaces at $m = 3, 4, 5, \dots$** What additional measurement would pick one,
   and why can the outboard probes not supply it?
8. **The steepest $dI_p/dt$ on 39915 changes by a factor of about 2.6 with smoothing, while the
   80–20 time is the same in every row.** Why is that sameness not evidence that the 80–20 time
   is robust, and which number must carry its smoothing width when you quote it?

## Integrated Analysis

### Part II, level 1 — the same discharge over time

Everything in Part I was written against the data rather than a typed time, so it can be walked
across `SHOT`: the tracked frequency, the midplane-trio $n$ at each tracked window, the band power,
and the Mirnov–SXR coherence in consecutive thirds of the plasma window.

In [ ]:
n_time, n_value, n_residual = [], [], []
if has_array:
    midplane_signals = np.array([np.asarray(ods[f"magnetics.b_field_pol_probe.{i}.voltage.data"], dtype=float)
                                 for i in midplane])
    midplane_phi = np.array([float(ods[f"magnetics.b_field_pol_probe.{i}.position.phi"]) for i in midplane])
    for when, frequency in zip(mode_track.time[tracked], mode_track.frequency[tracked]):
        fit = toroidal_phase_fit_at_time(mode_time, midplane_signals, midplane_phi, center_time=when,
                                         window_size=2000, frequencies=[frequency], num_modes=1,
                                         candidate_n=residues)
        n_time.append(when)
        n_value.append(fit.modes[0].n)
        n_residual.append(np.degrees(fit.modes[0].rms_error))

thirds = np.linspace(window[0], window[1], 4)
coherence_time, coherence_peak, coherence_line = [], [], []
if has_sxr:
    for start, stop in zip(thirds[:-1], thirds[1:]):
        pick_mode = (mode_time >= start) & (mode_time <= stop)
        pick_sxr = (sxr_time >= start) & (sxr_time <= stop)
        part = cross_spectrum(mode_time[pick_mode], mode_voltage[pick_mode], sxr_time[pick_sxr],
                              sxr_data[best, pick_sxr], nperseg=1024)
        branch = (part.frequency >= 2.0e3) & (part.frequency <= 1.0e4)
        coherence_time.append(0.5 * (start + stop))
        coherence_peak.append(part.coherence[branch].max())
        coherence_line.append(part.significance_95)

figure, axes = plt.subplots(4, 1, figsize=(9, 11), sharex=True)
axes[0].plot(mode_track.time * 1e3, mode_track.frequency / 1e3, "o-")
axes[0].set_ylabel("tracked f [kHz]")
axes[1].scatter(np.array(n_time) * 1e3, n_value, c=n_residual, cmap="viridis_r", vmin=0, vmax=45)
axes[1].set_ylabel(f"n (mod {array['alias_step']})\ncolour: rms [deg]")
axes[1].set_yticks(list(residues))
axes[2].semilogy(mode_evolution.time * 1e3, band_power)
axes[2].set_ylabel("2-40 kHz power [V$^2$]")
axes[3].plot(np.array(coherence_time) * 1e3, coherence_peak, "s-", label="max coherence 2-10 kHz")
axes[3].plot(np.array(coherence_time) * 1e3, coherence_line, "r--", label="95 % line")
axes[3].set_ylabel("Mirnov-SXR")
axes[3].legend(fontsize="small")
axes[3].set_xlabel("Time [ms]")
for axis in axes:
    axis.set_xlim(window[0] * 1e3, window[1] * 1e3)
    axis.grid(alpha=0.5)
axes[0].set_title(f"#{SHOT}: one discharge over time")
plt.show()
if n_value:
    counts = {value: n_value.count(value) for value in sorted(set(n_value))}
    print(f"n on the midplane trio over {len(n_value)} tracked windows: "
          + ", ".join(f"n = {value} in {count}" for value, count in counts.items())
          + f"; median residual {np.median(n_residual):.0f} deg")
print(f"rotation over time: not measured on shot {SHOT}")

The midplane $n$ holds while the frequency changes — $n = 1$ in all but one tracked window, the one
where the tracker jumped to a different line: the drop from the 7 kHz to the 3 kHz branch is not
accompanied by a change of toroidal structure on this trio. That favours one structure
changing its lab-frame frequency — a slowing plasma or a slowing mode — over a different mode
taking over, and it is still a statement about $n$ modulo 4 on three probes, with no rotation
measurement to say which slowed.

### Part II, level 2 — several discharges

The same measurements on a list of shots. The commented line is the lab-mode path for any list.
What each shot *supports* differs, and the table says so rather than filling the gap: only a shot
with a toroidal array gets an $n$.

In [ ]:
shots = [45531, 39915, 41524, 41672]
ods_list = [vaft.omas.sample_ods(shot) for shot in shots]
# ods_list = vaft.database.load(shots)  # lab mode: any list of shots, needs HSDS credentials
print(f"loaded {len(ods_list)} discharges: {shots}")

In [ ]:
columns = ("shot", "onset [ms]", "first line [ms]", "f median [kHz]", "power ratio", "n", "80-20 [ms]",
           "spike [kA]", "fluctuation records")
table = []
traces = []
for shot, case in zip(shots, ods_list):
    case_timing = plasma_timing(case)
    case_events = discharge_timing(case)
    names = [str(case[f"magnetics.b_field_pol_probe.{i}.name"]) for i in range(len(case["magnetics.b_field_pol_probe"]))]
    if eq_probe not in names:
        print(f"shot {shot}: no {eq_probe} -- no frequency track for this row")
        continue
    k = names.index(eq_probe)
    t = np.asarray(case[f"magnetics.b_field_pol_probe.{k}.voltage.time"], dtype=float)
    v = np.asarray(case[f"magnetics.b_field_pol_probe.{k}.voltage.data"], dtype=float)
    evolution = compute_spectrogram(t, v, window_duration=1.0e-3, overlap=0.5)
    track = track_dominant_frequency(evolution, search_range=(2.0e3, 4.0e4), max_jump=3.0e3)
    inside_case = (track.time >= case_timing.onset) & (track.time <= case_timing.offset) & np.isfinite(track.frequency)
    power = np.array([compute_band_power(evolution.frequency, column, {"mhd": (2.0e3, 4.0e4)})["mhd"]
                      for column in evolution.magnitude.T])
    case_floor_end = case_events.oh_onset if case_events.oh_onset is not None else case_timing.onset
    before_coils = power[evolution.time < case_floor_end]
    floor = np.median(before_coils) if before_coils.size else np.nan
    active_power = np.median(power[(evolution.time >= case_timing.onset) & (evolution.time <= case_timing.offset)])
    row = next(r for r in vaft.omas.available_plots(case, available_only=False) if r["name"] == "mirnov_spatial_phase")
    if row["available"]:
        n_text = f"n={trio_fits['L1-03'].n} mod {array['alias_step']}" if shot == SHOT and has_array and "L1-03" in trio_fits else "fit allowed"
    else:
        n_text = "not allowed"
    case_time = np.asarray(case["magnetics.ip.0.time"], dtype=float)
    case_ip = np.asarray(case["magnetics.ip.0.data"], dtype=float)
    case_quench = current_quench(case_time, case_ip)
    case_spike = (current_spike(case_time, case_ip, before=case_quench.time_20, lookback_s=case_quench.duration_80_20)
                  if case_quench.found else None)
    table.append([
        f"{shot}", f"{case_timing.onset * 1e3:.1f}",
        f"{track.time[inside_case][0] * 1e3:.1f}" if inside_case.any() else "--",
        f"{np.median(track.frequency[inside_case]) / 1e3:.1f}" if inside_case.any() else "--",
        f"{active_power / floor:.0f}x" if np.isfinite(floor) and floor > 0 else "no floor",
        n_text,
        f"{case_quench.duration_80_20 * 1e3:.1f}" if case_quench.found else "no quench",
        f"+{case_spike.amplitude / 1e3:.1f}" if case_spike is not None and case_spike.time is not None else "--",
        ", ".join(sorted({label.split(" /")[0].split(" (")[0] for label in vaft.omas.fluctuation_bandwidths(case)})),
    ])
    traces.append((shot, case_time - case_timing.onset, np.abs(case_ip), track.time - case_timing.onset,
                   track.frequency, case_timing.offset - case_timing.onset))

widths = [max(len(name), *(len(row[i]) for row in table)) for i, name in enumerate(columns)]
print("  ".join(name.rjust(width) for name, width in zip(columns, widths)))
for row in table:
    print("  ".join(cell.rjust(width) for cell, width in zip(row, widths)))

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for shot, t_ip, current, t_track, frequency, _duration in traces:
    axes[0].plot(t_ip * 1e3, current / 1e3, label=f"#{shot}")
    axes[1].plot(t_track * 1e3, frequency / 1e3, "o-", ms=3, label=f"#{shot}")
axes[0].set_ylabel("|Ip| [kA]")
axes[1].set_ylabel(f"tracked f, {eq_probe.split('_')[1]} [kHz]")
axes[1].set_xlabel("Time from each shot's plasma onset [ms]")
axes[1].set_xlim(-5.0, max(trace[-1] for trace in traces) * 1e3 + 5.0)
axes[0].set_ylim(bottom=0.0)
for axis in axes:
    axis.legend(fontsize="small")
    axis.grid(alpha=0.5)
plt.show()

Read the table with its caveats attached:

- **Times are aligned to each shot's own plasma onset**, found by the same helper — never to a
  shared clock time.
- **The frequency column is the same outboard equilibrium probe on every shot**, 250 kHz, so the
  shots are compared on one instrument even where 45531 has better ones.
- **$n$ is "not allowed" wherever no toroidal array recorded**, which is every shot but 45531.
  A missing $n$ is an entry, not a blank to fill from a similar-looking spectrogram.
- **The 80–20 time and the spike** are measurements of the current, not of the mode; the quench
  on a shot with no coherent line is still a quench, and nothing in the row names the event.

## Independent Exercise

### A fluctuation-rich discharge of your choice

Pick a VEST discharge (lab mode, `vaft.database.load`) — ideally from shot 44156 on, when the
outboard fluctuation array exists, and with soft X-ray or camera data — and write a short report:

1. **The usable plasma window**, from `plasma_timing` and `discharge_timing`.
2. **Sampling rate and usable bandwidth** of each diagnostic you use, from
   `fluctuation_bandwidths`, and what limits the usable band below Nyquist.
3. **A coherent Mirnov feature**: which probe, which band, and its floor.
4. **Track $f(t)$** with `track_dominant_frequency`, and say where the ridge is and is not defined.
5. **Estimate $n$** where the toroidal geometry permits, on every trio, with the residual.
6. **State the aliasing and geometry limits**: distinct positions, alias step, candidate ties.
7. **Search for the same feature** in SXR or another diagnostic.
8. **Coincidence or coherence?** Report coherence with its segment count and 95 % line, and the
   phase where it is significant.
9. **Lab or plasma frame?** If $v_\phi$ is measured on your shot, compute $n f_\phi$ and discuss
   $f_{\mathrm{plasma}}$; if not, say what the Doppler term could be.
10. **Connect to $q$** — the surfaces your $n$ allows — and state the limitation of the missing $m$.
11. **Classify the observation** as a coherent mode, broadband activity or a nonlinear transient,
    using only the evidence you measured, and name nothing a function did not decide.

The first two lines of the cell are also the place to take the array further on 45531: repeat the
trio fits of step 1.7 at other times, and decide why the off-midplane trios disagree.

In [ ]:
# Lab mode: needs HSDS credentials. Replace the shot with the one you chose.
# SHOT = <your shot>
# Swap the lab-mode line in the Load / Prepare Data cell, then rerun Part I from the top:
# the window, the tracked frequency and every step after them follow the new shot.
#
# 2. bandwidths
# for label, (rate, nyquist) in vaft.omas.fluctuation_bandwidths(ods).items():
#     print(label, rate, nyquist)
#
# 4. the tracked frequency of any probe you choose, by name
# probe = probe_names.index("<probe name>")
# t = np.asarray(ods[f"magnetics.b_field_pol_probe.{probe}.voltage.time"])
# v = np.asarray(ods[f"magnetics.b_field_pol_probe.{probe}.voltage.data"])
# track = track_dominant_frequency(compute_spectrogram(t, v, window_duration=1.0e-3),
#                                  search_range=(2.0e3, 4.0e4), max_jump=3.0e3)
#
# 5-6. every trio of the fluctuation array, at one tracked time
# for row in ("L1-01", "L1-03", "L1-05", "L2-01", "L2-03", "L2-05"):
#     trio = [probe_names.index(f"OutMirnov_{clock}_{row}") for clock in (45, 135, 225)]
#     vaft.omas.plot_mirnov_spatial_phase(ods, time=t_mode, channels=trio, num_modes=1,
#                                         candidate_n=range(-2, 3), window_size=2000)
#     plt.show()
#
# 8. coherence of any two channels, with its 95 % line
# vaft.omas.plot_diagnostics_spectrum_coherence(ods, x_signal="mirnov:<probe>", y_signal="sxr:<chord>",
#                                               time_range=window, nperseg=2048)
#
# Camera frames for a database shot are a separate source:
# camera_ods = vaft.database.load(<shot>, source="camera-visible-fluctuation")
#
# A third transform, through the optional fcwt package (method="cwt" needs frequency_range=):
# vaft.omas.plot_mirnov_spectrogram(ods, channel=mode_index, method="cwt", time_range=window,
#                                   frequency_range=(2.0e3, 5.0e4), n_frequencies=200)

## Takeaways and Next Steps

- A perturbation's frequency and its mode numbers are two projections of one rotating structure.
  A fixed probe measures the first; an array measures the second, **modulo its alias step**.
- $f_{\mathrm{lab}} \approx f_{\mathrm{plasma}} + n f_\phi$, and on VEST the Doppler term for
  $n = 1$ is the same order of magnitude as the observed lines. An observed frequency is not an
  intrinsic one until rotation is measured in the same discharge.
- Diagnostics measure different variables — $dB/dt$, line-integrated emissivity, projected light,
  line-integrated density, edge line emission — each with a usable band below its Nyquist
  frequency. Sampling rate is never detectability.
- Coincidence is weak evidence; coherence above its 95 % line, with a stable phase and a stated
  segment count, is strong. The Mirnov–SXR and probe–camera coherences are what tie the three
  diagnostics to one perturbation.
- A fit asked for $n$ returns one. The residual, the number of distinct positions and the
  agreement between trios are what make it a measurement — and a plot that refuses, with its
  reason, is better than one that returns an identity (#724, #825).
- $n$ alone does not fix a resonant surface: $m$ is needed and VEST's outboard probes cannot
  supply it (#506, #472); the mode–rotation overlay is #460.
- Transients are sequences of measured numbers — onset, quench time, spike, collapse — and VAFT
  has no classifier that names them. Quote a steepest slope with its smoothing, and do not
  differentiate a reconstruction's artefacts.

**Method depth**: `notebooks/fluctuation_diagnostics_analysis.ipynb`.

**Next**: Session 05 stops measuring what fluctuated and asks what *could*: it takes the
equilibrium into linear stability and perturbed-equilibrium calculations — tearing, ideal and
3-D response — where $m$, $n$ and the rational surfaces of this session become inputs rather than
unknowns.